# Fly trajectory pattern analysis

**Workflow (run top to bottom):**

1. **Load / Setup** — imports + load `merged_detection_all.csv`, compute kinematics.
   *First run `python merge_detection_csvs.py` in **this folder** (`Exp1`) to generate `merged_detection_all.csv`.*

2. **Step 0 — global pattern discovery** — pool every trajectory from the merged CSV and determine how many distinct movement patterns (k) best describe the data at each segment length (5–40 mm). Outputs a summary table and plots to `pattern_distance_output/0_global/`. **Figures** label cohorts **YM, YF, OM, OF** (young/old × male/female) when the merged CSV includes `sex` and `age_group`.

3. **Steps 1–3** — detailed analyses using the patterns found in Step 0:
   - **Step 1** — per fly / per CSV: same **8 figures per segment-length folder** as Step 0
   - **Step 2** — all males pooled and all females pooled (young vs old)
   - **Step 3** — young and old pooled (male vs female)


In [3]:
import sys
from pathlib import Path
from typing import Optional

# Expect this repo's `.venv` (requirements.txt). Wrong kernel → missing seaborn/sklearn, etc.
def _project_venv_python(start: Path) -> Optional[Path]:
    for d in [start, *start.parents][:12]:
        cand = d / ".venv" / "bin" / "python"
        if cand.is_file():
            return cand.resolve()
    return None


_root = Path.cwd()
_venv_py = _project_venv_python(_root)
if _venv_py is not None and Path(sys.executable).resolve() != _venv_py:
    raise RuntimeError(
        "Wrong Python interpreter for this notebook.\n"
        f"  Currently using: {sys.executable}\n"
        f"  Use instead:     {_venv_py}\n\n"
        "In Cursor: kernel picker (top-right) → **Python (.venv Experiment1)**. "
        "Or Command Palette → **Notebook: Select Notebook Kernel**.\n"
        "If that kernel is missing, in Terminal:\n"
        f"  cd {_venv_py.parent.parent.parent} && python3 -m venv .venv && "
        ".venv/bin/pip install -r requirements.txt && .venv/bin/python -m ipykernel install --user "
        "--name=experiment1-pattern --display-name='Python (.venv Experiment1)'"
    )

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from typing import Dict, Tuple

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
import warnings
warnings.filterwarnings("ignore")

sns.set_context("talk")
sns.set_style("whitegrid")

# Default: merged WIG trajectories from merge_detection_csvs.py (run Jupyter with cwd = this folder, or set full path).
_merged = Path.cwd() / "merged_detection_all.csv"
CSV_PATH = str(_merged) if _merged.is_file() else "merged_detection_all.csv"
# Other examples: "merged_kw_detection_output.csv", or an absolute path string.
USER_VIDEO_METADATA_CSV: Optional[str] = None  # optional overrides per (experiment_id, detection_run_id)
DEFAULT_MM_PER_PX_X = 0.04
DEFAULT_MM_PER_PX_Y = 0.05
MM_PER_PX_BY_RUN: Dict[Tuple[str, int], Tuple[float, float]] = {}

PLOT_DIR = Path("pattern_distance_output")
PLOT_DIR.mkdir(parents=True, exist_ok=True)

from pattern_export_stats import export_pattern_feature_stats_csv, export_pca_specification_csv

# SKIP_PATTERN_FIGURES — True = skip regenerating pattern PNGs only; clustering unchanged.
SKIP_PATTERN_FIGURES = False
# If True: only write cluster_naming_order.csv (skip segment_measures & other exports). Clustering time unchanged.
ONLY_CLUSTER_NAMING_ORDER_CSV = False


def save_fig(name: str, fig=None):
    if fig is None:
        fig = plt.gcf()
    out = PLOT_DIR / name
    fig.savefig(out, dpi=200, bbox_inches="tight")
    return out

FEATURE_COLS = ["straightness", "tortuosity", "path_per_40", "mean_abs_turn_deg", "n_turns"]
# Turn threshold (deg): direction change above this counts as a turn. Literature: 10-15° conservative; gait/locomotion often 22-45°.
TURN_THRESHOLD_DEG = 15
def turn_stats_from_xy(x, y):
    """From (x, y) points, return (mean_abs_turn_deg, n_turns). n_turns = count of direction changes > TURN_THRESHOLD_DEG."""
    x, y = np.asarray(x, dtype=float), np.asarray(y, dtype=float)
    if len(x) < 3:
        return np.nan, 0
    dx = np.diff(x)
    dy = np.diff(y)
    head = np.arctan2(dy, dx)
    turn = np.diff(head)
    turn = np.where(turn > np.pi, turn - 2 * np.pi, np.where(turn < -np.pi, turn + 2 * np.pi, turn))
    deg = np.degrees(np.abs(turn))
    mean_deg = np.nanmean(deg) if len(deg) else np.nan
    n_turns = np.sum(deg > TURN_THRESHOLD_DEG) if len(deg) else 0
    return mean_deg, int(n_turns)


def apply_user_video_metadata(videos: pd.DataFrame, meta_path: Optional[str]) -> pd.DataFrame:
    """Merge user-supplied age/sex/etc. on (experiment_id, detection_run_id). User non-null values override code-derived fields."""
    if meta_path is None or not Path(meta_path).is_file():
        return videos
    um = pd.read_csv(meta_path)
    um["detection_run_id"] = um["detection_run_id"].astype(int)
    vc = ["experiment_id", "detection_run_id"]
    if not set(vc).issubset(um.columns):
        raise ValueError(f"User metadata CSV must include columns: {vc}")
    if "gender" in um.columns and "sex" not in um.columns:
        um = um.rename(columns={"gender": "sex"})
    v = videos.set_index(vc)
    u = um.set_index(vc)
    for col in u.columns:
        if col in v.columns:
            v[col] = u[col].combine_first(v[col])
        else:
            v[col] = u[col]
    return v.reset_index()


def _recategory_from_row(row) -> str:
    ad, g = row.get("age_days"), row.get("genotype")
    if pd.notna(ad) and pd.notna(g) and str(g) not in ("nan", "None", ""):
        try:
            return f"age{int(float(ad))}_{g}"
        except (ValueError, TypeError):
            pass
    c = row.get("category")
    return c if pd.notna(c) and str(c) not in ("nan", "") else "unknown"


def _age_group_from_row(row) -> str:
    ag = row.get("age_group")
    if pd.notna(ag) and str(ag).strip() not in ("", "nan"):
        return str(ag).strip()
    c = row.get("category")
    if c and "age11" in str(c):
        return "young"
    if c and "age52" in str(c):
        return "old"
    return "unknown"


## Load / Setup

Reads `merged_detection_all.csv`, normalises column names, computes `dt_s`, `step_dist_mm`, `cum_dist_mm`, and `total_dist_mm` for every `(experiment_id, detection_run_id)`. Run this cell **before any Step**.


In [4]:
# Load / Setup: load merged CSV (merged_detection_all.csv); normalize column names from merge_detection_csvs.py
_csv_path = Path(CSV_PATH)
if not _csv_path.is_file():
    _csv_path = Path.cwd() / Path(CSV_PATH).name
if not _csv_path.is_file():
    raise FileNotFoundError(
        f"CSV not found: {CSV_PATH!r}. Set CSV_PATH in the imports cell, set Jupyter cwd to this notebook folder (Exp1), or run: python merge_detection_csvs.py"
    )

df = pd.read_csv(_csv_path)
_rename = {}
if "experiment" in df.columns and "experiment_id" not in df.columns:
    _rename["experiment"] = "experiment_id"
if "fly_id" in df.columns and "detection_run_id" not in df.columns:
    _rename["fly_id"] = "detection_run_id"
if _rename:
    df = df.rename(columns=_rename)
if "gender" in df.columns and "sex" not in df.columns:
    df = df.rename(columns={"gender": "sex"})

required = {"experiment_id", "detection_run_id", "frame", "time_s", "x", "y"}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {sorted(missing)}")

df["detection_run_id"] = pd.to_numeric(df["detection_run_id"], errors="coerce")
if df["detection_run_id"].isna().any():
    raise ValueError("detection_run_id has invalid (non-numeric) values")
df["detection_run_id"] = df["detection_run_id"].astype(int)

video_cols = ["experiment_id", "detection_run_id"]
videos = df[video_cols].drop_duplicates().copy()

# True when using merged_detection_all.csv (per-row age / sex / genotype)
_use_merged = "sex" in df.columns and ("age" in df.columns or "age_group" in df.columns)

if _use_merged:
    _agg_cols = [c for c in ("age", "genotype", "sex", "category", "age_group") if c in df.columns]
    per_vid = df.groupby(video_cols, as_index=False)[_agg_cols].first() if _agg_cols else videos.copy()
    videos = videos.merge(per_vid, on=video_cols, how="left")
    if "age_group" not in videos.columns and "age" in videos.columns:
        videos["age_group"] = videos["age"].astype(str).str.strip().str.lower()
    if "category" not in videos.columns or videos["category"].isna().all():
        if "genotype" in videos.columns and "age_group" in videos.columns:
            videos["category"] = videos["genotype"].astype(str) + "_" + videos["age_group"].astype(str)
        else:
            videos["category"] = "unknown"
    videos["age_days"] = np.nan
else:
    videos["age_days"] = None
    videos["genotype"] = None
    videos["category"] = "all"

videos = apply_user_video_metadata(videos, USER_VIDEO_METADATA_CSV)
videos["category"] = videos.apply(_recategory_from_row, axis=1)
videos["age_group"] = videos.apply(_age_group_from_row, axis=1)

cal = videos.apply(
    lambda r: MM_PER_PX_BY_RUN.get((r["experiment_id"], int(r["detection_run_id"])), (DEFAULT_MM_PER_PX_X, DEFAULT_MM_PER_PX_Y)),
    axis=1,
)
videos["mm_per_px_x"] = [c[0] for c in cal]
videos["mm_per_px_y"] = [c[1] for c in cal]

merge_cols = video_cols + ["age_days", "genotype", "category", "mm_per_px_x", "mm_per_px_y", "age_group"]
for _opt in ("sex", "age"):
    if _opt in videos.columns:
        merge_cols.append(_opt)
_overlap = [c for c in merge_cols if c in df.columns and c not in video_cols]
if _overlap:
    df = df.merge(videos[merge_cols], on=video_cols, how="left", suffixes=("", "_video_meta"))
    for _c in _overlap:
        cm = f"{_c}_video_meta"
        if cm in df.columns:
            df[_c] = df[cm].combine_first(df[_c])
            df.drop(columns=[cm], inplace=True)
else:
    df = df.merge(videos[merge_cols], on=video_cols, how="left")
df = df.sort_values(video_cols + ["frame", "time_s"]).reset_index(drop=True)

g = df.groupby(video_cols, sort=False)
df["dt_s"] = g["time_s"].diff()
df["dx_px"] = g["x"].diff()
df["dy_px"] = g["y"].diff()
df.loc[df["dt_s"] == 0, "dt_s"] = np.nan

dx_mm = df["dx_px"] * df["mm_per_px_x"]
dy_mm = df["dy_px"] * df["mm_per_px_y"]
df["step_dist_mm"] = np.sqrt(dx_mm**2 + dy_mm**2).fillna(0.0)
df["speed_mm_s"] = df["step_dist_mm"] / df["dt_s"]
df["cum_dist_mm"] = g["step_dist_mm"].cumsum()

total = (
    df.groupby(video_cols, as_index=False)["cum_dist_mm"].max()
    .rename(columns={"cum_dist_mm": "total_dist_mm"})
)
df = df.merge(total, on=video_cols, how="left")

# Fill any missing age_group from category (age11/age52 wig convention)
_fill_ag = df["category"].map(
    lambda c: "young" if c and "age11" in str(c) else ("old" if c and "age52" in str(c) else "unknown")
)
df["age_group"] = df["age_group"].combine_first(_fill_ag)

print(f"Rows: {len(df):,}, Videos: {len(total):,}")
print("Categories:", df["category"].value_counts().to_dict())
print("Age groups:", df["age_group"].value_counts().to_dict())
df.head(3)


Rows: 2,749,007, Videos: 65
Categories: {'WIG_young': 2195953, 'WIG_old': 553054}
Age groups: {'young': 2195953, 'old': 553054}


,experiment_id,detection_run_id,frame,time_s,x,y,w,h,area,age,...,mm_per_px_x,mm_per_px_y,age_group,dt_s,dx_px,dy_px,step_dist_mm,speed_mm_s,cum_dist_mm,total_dist_mm
0,sd_09_12,1,31,0.534483,319,134,28,24,478,young,...,0.04,0.05,young,NaN,NaN,NaN,0.00,NaN,0.00,7999.888697
1,sd_09_12,1,32,0.551724,319,133,28,26,530,young,...,0.04,0.05,young,0.017241,0.0,-1.0,0.05,2.900064,0.05,7999.888697
2,sd_09_12,1,33,0.568966,319,132,28,27,515,young,...,0.04,0.05,young,0.017242,0.0,-1.0,0.05,2.899896,0.10,7999.888697


## Steps 0 – 3

**Step 0. Global pooled** — entire `df`, all flies: find how many movement patterns (k) exist per segment length. Summary → `0_global/step0_pattern_summary.csv`.

**Step 1. Per CSV / per fly** — each `(experiment_id, detection_run_id)` is one detection file. **Inside every** `1_entire_data/<exp>_run_<id>/<N>mm/` folder you get **8 PNGs** (mirroring Step 0): `step1_silhouette_k.png`, `step1_pattern_counts.png`, `step1_pca_patterns.png`, `step1_movements_per_pattern.png`, `step1_movements_avg_turn_per_pattern.png`, `step1_movements_n_turns_per_pattern.png`, `step1_movements_turn_angle_vs_n_turns.png`, `step1_actual_positions.png`. Six distance folders × 8 PNGs per fly when all lengths run; plus `step1_pattern_summary.csv` at `.../<exp>_run_<id>/`.

**Step 2. Males / females** — pool all young+old males, then all females. Output: `2_males/<mm>mm/`, `2_females/<mm>mm/`

**Step 3. Young / old** — male vs female within young, within old. Output: `3_young/<mm>mm/`, `3_old/<mm>mm/`

**Cohort experiments (one cell)** — runs all five comparisons at once: young vs old (all sexes / males only / females only), male vs female (young / old). Output: `cohort_experiments/<experiment_key>/`.


## Step 0 — Global pattern discovery

Pools **all** rows from `merged_detection_all.csv` into one analysis.
For each segment length (5, 10, 15, 20, 30, 40 mm):
1. Splits every trajectory into fixed-distance segments and computes 5 shape features
   (straightness, tortuosity, path ratio, mean turn angle, turn count).
2. Runs k-means for k = 2…10 and picks **k** with the best silhouette score.
3. Labels each cluster (straight, zig-zag, curved, meandering, …).
4. Saves `pattern_distance_output/0_global/step0_pattern_summary.csv` + **8** figures per segment length (including three movement stat PNGs).

**Cohort labels (merged CSV with `sex` + `age_group`):** **YM** = young male, **YF** = young female, **OM** = old male, **OF** = old female (distinct colours). If `sex` is missing, those segments are drawn in light gray and omitted from the 4-way boxplots; if the merged file has no `sex` column, figures fall back to young vs old only.

**Run Step 0 first** to learn k for your data before running Steps 1–3.


In [ ]:
# Step 0: Pool ALL merged trajectories — discover how many movement patterns (k) exist at each segment length.
# Same segment features and silhouette-based k as Type 1, but one single pass over the entire df.
try:
    from IPython.display import display
except ImportError:
    display = print
STEP0_DISTANCES_MM = [5, 10, 15, 20, 30, 40]
MIN_PATH_MM = 5.0
BASE0 = Path("pattern_distance_output") / "0_global"
BASE0.mkdir(parents=True, exist_ok=True)

df_sub = df.dropna(subset=["cum_dist_mm", "total_dist_mm"]).copy()
df_sub = df_sub[df_sub["total_dist_mm"] > 0].copy()
if len(df_sub) == 0:
    raise ValueError("Step 0: no rows after filtering cum_dist / total_dist")

summary_rows = []

for seg_mm in STEP0_DISTANCES_MM:
    out_dir = BASE0 / f"{seg_mm}mm"
    out_dir.mkdir(parents=True, exist_ok=True)
    old_plot_dir = PLOT_DIR
    globals()["PLOT_DIR"] = out_dir
    SEGMENT_LENGTH_MM = seg_mm

    df_seg = df_sub.copy()
    if "sex" in df_seg.columns:
        df_seg["sex"] = df_seg["sex"].astype(str).str.strip().str.lower()
        df_seg.loc[df_seg["sex"].isin(["nan", "none", ""]), "sex"] = np.nan
    df_seg["seg_40"] = (df_seg["cum_dist_mm"] // SEGMENT_LENGTH_MM).astype(int) + 1
    n_segs_per_video = (
        df_seg.groupby(video_cols)["total_dist_mm"].transform("first") / SEGMENT_LENGTH_MM
    ).apply(np.ceil).astype(int)
    df_seg["seg_40"] = df_seg["seg_40"].clip(upper=n_segs_per_video)

    seg = (
        df_seg.groupby(video_cols + ["category", "age_group", "seg_40"], as_index=False)
        .agg(
            n_rows=("frame", "size"),
            t_start_s=("time_s", "min"),
            t_end_s=("time_s", "max"),
            seg_path_mm=("step_dist_mm", "sum"),
            speed_mean_mm_s=("speed_mm_s", "mean"),
            x_start=("x", "first"),
            y_start=("y", "first"),
            x_end=("x", "last"),
            y_end=("y", "last"),
        )
    )
    seg["seg_dur_s"] = seg["t_end_s"] - seg["t_start_s"]
    seg["net_dx_px"] = seg["x_end"] - seg["x_start"]
    seg["net_dy_px"] = seg["y_end"] - seg["y_start"]
    seg["net_disp_px"] = np.sqrt(seg["net_dx_px"] ** 2 + seg["net_dy_px"] ** 2)
    cal = df_seg.groupby(video_cols, as_index=False).agg(
        mm_per_px_x=("mm_per_px_x", "first"),
        mm_per_px_y=("mm_per_px_y", "first"),
    )
    seg = seg.merge(cal, on=video_cols, how="left")
    _vid_sex = df_seg.groupby(video_cols, as_index=False)["sex"].first()
    seg = seg.merge(_vid_sex, on=video_cols, how="left")
    seg["net_disp_mm"] = np.sqrt(
        (seg["net_dx_px"] * seg["mm_per_px_x"]) ** 2 + (seg["net_dy_px"] * seg["mm_per_px_y"]) ** 2
    )
    seg["straightness"] = seg["net_disp_mm"] / seg["seg_path_mm"].replace({0: np.nan})
    seg = seg[seg["seg_path_mm"] >= MIN_PATH_MM].copy()
    seg["tortuosity"] = seg["seg_path_mm"] / seg["net_disp_mm"].replace(0, np.nan).clip(lower=1e-6)
    seg["path_per_40"] = seg["seg_path_mm"] / SEGMENT_LENGTH_MM

    mean_turns, n_turns_list = [], []
    for _, r in seg.iterrows():
        sub = df_seg[
            (df_seg["experiment_id"] == r["experiment_id"])
            & (df_seg["detection_run_id"] == r["detection_run_id"])
            & (df_seg["seg_40"] == r["seg_40"])
        ].sort_values(["frame", "time_s"])
        ma, nt = turn_stats_from_xy(sub["x"].values, sub["y"].values)
        mean_turns.append(ma)
        n_turns_list.append(nt)
    seg["mean_abs_turn_deg"] = mean_turns
    seg["n_turns"] = n_turns_list
    seg = seg.dropna(subset=["straightness", "tortuosity", "path_per_40"]).copy()
    seg["mean_abs_turn_deg"] = seg["mean_abs_turn_deg"].fillna(0)

    if len(seg) < 10:
        print(f"  Step 0 {seg_mm}mm: too few segments ({len(seg)}), skip")
        globals()["PLOT_DIR"] = old_plot_dir
        continue

    X = seg[FEATURE_COLS].copy()
    _feat_scaler = StandardScaler()
    X_scaled = _feat_scaler.fit_transform(X)
    K_RANGE = range(2, max(3, min(11, len(seg) // 5)))
    silhouettes, inertias = [], []
    for k in K_RANGE:
        km_trial = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X_scaled)
        silhouettes.append(silhouette_score(X_scaled, km_trial.predict(X_scaled)))
        inertias.append(km_trial.inertia_)
    best_k = list(K_RANGE)[np.argmax(silhouettes)] if silhouettes else 2
    n_clusters = best_k
    km = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    seg["pattern"] = km.fit_predict(X_scaled)
    seg["pattern"] = seg["pattern"].astype(int)
    seg_clean = seg.copy()

    means = seg_clean.groupby("pattern")[FEATURE_COLS].mean()
    patterns = sorted(means.index.tolist())
    assigned = set()

    def assign_best(lbl, feature, order="max"):
        remaining = [p for p in patterns if p not in assigned]
        if not remaining:
            return None
        pat = (
            means.loc[remaining, feature].idxmax()
            if order == "max"
            else means.loc[remaining, feature].idxmin()
        )
        assigned.add(pat)
        return pat

    label_specs = [
        ("zig-zag", "n_turns", "max"),
        ("straight", "straightness", "max"),
        ("meandering", "path_per_40", "max"),
        ("curved", "mean_abs_turn_deg", "max"),
        ("winding", "tortuosity", "max"),
        ("direct", "straightness", "max"),
        ("exploratory", "path_per_40", "max"),
    ]
    pattern_labels = {}
    for lab, feat, ord_ in label_specs[:n_clusters]:
        p = assign_best(lab, feat, ord_)
        if p is not None:
            pattern_labels[p] = lab
    for p in patterns:
        if p not in pattern_labels:
            pattern_labels[p] = "moderate"
    seg_clean["pattern_label"] = seg_clean["pattern"].map(pattern_labels)
    PATTERN_GROUP = {
        "straight": "Straight",
        "direct": "Direct",
        "meandering": "Meandering",
        "zig-zag": "Zig-zag / reversing",
        "curved": "Curved / turning",
        "winding": "Winding",
        "exploratory": "Exploratory",
        "moderate": "Other / moderate",
    }
    seg_clean["pattern_group"] = seg_clean["pattern_label"].map(PATTERN_GROUP)

    # Four-way cohort (global Step 0): YM/YF/OM/OF — needs merged CSV with sex + age_group
    _COHORT_ORDER = ["YM", "YF", "OM", "OF"]
    _COHORT_PALETTE = {"YM": "#1f77b4", "YF": "#2ca02c", "OM": "#ff7f0e", "OF": "#d62728"}
    _ag = seg_clean["age_group"].astype(str).str.strip().str.lower()
    if "sex" in seg_clean.columns:
        _sx = seg_clean["sex"].astype(str).str.strip().str.lower()
        _sx = _sx.replace({"nan": np.nan, "none": np.nan, "": np.nan})
    else:
        _sx = pd.Series(np.nan, index=seg_clean.index)
    seg_clean["cohort"] = np.nan
    seg_clean.loc[_ag.eq("young") & _sx.eq("male"), "cohort"] = "YM"
    seg_clean.loc[_ag.eq("young") & _sx.eq("female"), "cohort"] = "YF"
    seg_clean.loc[_ag.eq("old") & _sx.eq("male"), "cohort"] = "OM"
    seg_clean.loc[_ag.eq("old") & _sx.eq("female"), "cohort"] = "OF"
    _use_four_cohorts = bool(seg_clean["cohort"].notna().any())

    distinct_names = sorted(set(seg_clean["pattern_label"].astype(str)))
    best_sil = float(max(silhouettes)) if silhouettes else float("nan")
    summary_rows.append(
        {
            "segment_length_mm": seg_mm,
            "n_segments": len(seg_clean),
            "n_patterns_k": int(n_clusters),
            "n_distinct_labels": len(distinct_names),
            "best_silhouette": best_sil,
            "labels": "; ".join(distinct_names),
        }
    )
    print(
        f"  Step 0 {seg_mm}mm: n_segments={len(seg_clean):,}  ->  k={n_clusters} patterns  "
        f"(distinct labels: {len(distinct_names)})  silhouette={best_sil:.4f}"
    )


    export_pattern_feature_stats_csv(
        seg_clean,
        FEATURE_COLS,
        out_dir,
        segment_length_mm=seg_mm,
        step_name="step0_global",
        only_cluster_naming_order_csv=ONLY_CLUSTER_NAMING_ORDER_CSV,
    )

    _pca = PCA(n_components=2, random_state=42)
    pc_xy = _pca.fit_transform(X_scaled)
    export_pca_specification_csv(
        _feat_scaler,
        _pca,
        FEATURE_COLS,
        out_dir,
        analysis_id="step0_global",
        segment_length_mm=seg_mm,
        n_segments=len(seg_clean),
        k_patterns=int(n_clusters),
    )

    if not SKIP_PATTERN_FIGURES:
        # ── Plot 1: silhouette + inertia ────────────────────────────────────────
        fig, ax = plt.subplots(1, 2, figsize=(10, 4))
        ax[0].plot(list(K_RANGE), silhouettes, "o-")
        ax[0].axvline(n_clusters, color="green", linestyle="--", alpha=0.8, label=f"chosen k={n_clusters}")
        ax[0].set_xlabel("k (number of patterns)")
        ax[0].set_ylabel("Silhouette score")
        ax[0].legend()
        ax[1].plot(list(K_RANGE), inertias, "o-")
        ax[1].axvline(n_clusters, color="green", linestyle="--", alpha=0.8)
        ax[1].set_xlabel("k")
        ax[1].set_ylabel("Inertia")
        plt.suptitle(f"Step 0 global pooled — {seg_mm} mm segments", y=1.02)
        plt.tight_layout()
        save_fig("step0_silhouette_k.png")
        plt.close()

        # ── Plot 2: bar chart — segments per pattern ────────────────────────────
        fig, ax = plt.subplots(figsize=(9, 4))
        seg_clean["pattern_label"].value_counts().sort_index().plot(kind="bar", ax=ax, color="steelblue", edgecolor="gray")
        ax.set_ylabel("Segment count")
        ax.set_xlabel("Pattern label")
        ax.set_title(f"Step 0: segments per pattern (k={n_clusters}, {seg_mm} mm)")
        plt.xticks(rotation=25, ha="right")
        plt.tight_layout()
        save_fig("step0_pattern_counts.png")
        plt.close()

        # ── Plot 3: PCA scatter coloured by pattern ─────────────────────────────
        seg_clean["PC1"] = pc_xy[:, 0]
        seg_clean["PC2"] = pc_xy[:, 1]
        fig, ax = plt.subplots(figsize=(10, 6.5))
        sns.scatterplot(data=seg_clean, x="PC1", y="PC2", hue="pattern_label", alpha=0.25, s=8, ax=ax)
        ax.set_title(f"Step 0 PCA — {seg_mm} mm (k={n_clusters})")
        ax.legend(
            bbox_to_anchor=(0.5, -0.14),
            loc="upper center",
            ncol=3,
            fontsize=8,
            title="Pattern",
            title_fontsize=9,
            frameon=True,
            columnspacing=0.9,
            handletextpad=0.4,
        )
        fig.tight_layout()
        fig.subplots_adjust(bottom=0.24)
        save_fig("step0_pca_patterns.png")
        plt.close()

        # ── Plot 4: trajectories per pattern (colour = age+sex cohorts when merged CSV has sex) ──
        groups_sorted = seg_clean["pattern_group"].dropna().unique()
        groups_sorted = sorted(groups_sorted, key=lambda g: -(seg_clean["pattern_group"] == g).sum())
        n_groups = len(groups_sorted)
        n_cols = min(3, max(1, n_groups))
        n_rows = int(np.ceil(n_groups / n_cols)) if n_groups else 1
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 5 * n_rows))
        axes = np.atleast_2d(axes)

        for idx_g, grp in enumerate(groups_sorted):
            ax = axes.flat[idx_g]
            subset = seg_clean[seg_clean["pattern_group"] == grp]
            if _use_four_cohorts:
                for c in _COHORT_ORDER:
                    sub = subset[subset["cohort"] == c]
                    if len(sub) == 0:
                        continue
                    for _, r in sub.iterrows():
                        pts = df_seg[
                            (df_seg["experiment_id"] == r["experiment_id"])
                            & (df_seg["detection_run_id"] == r["detection_run_id"])
                            & (df_seg["seg_40"] == r["seg_40"])
                        ].sort_values(["frame", "time_s"])
                        x = pts["x"].values.astype(float) - pts["x"].iloc[0]
                        y = pts["y"].values.astype(float) - pts["y"].iloc[0]
                        ax.plot(x, y, color=_COHORT_PALETTE[c], alpha=0.20, linewidth=0.7)
                sub_u = subset[subset["cohort"].isna()]
                for _, r in sub_u.iterrows():
                    pts = df_seg[
                        (df_seg["experiment_id"] == r["experiment_id"])
                        & (df_seg["detection_run_id"] == r["detection_run_id"])
                        & (df_seg["seg_40"] == r["seg_40"])
                    ].sort_values(["frame", "time_s"])
                    x = pts["x"].values.astype(float) - pts["x"].iloc[0]
                    y = pts["y"].values.astype(float) - pts["y"].iloc[0]
                    ax.plot(x, y, color="#999999", alpha=0.12, linewidth=0.6)
                _parts = [f"{c}: {(subset['cohort']==c).sum()}" for c in _COHORT_ORDER if (subset['cohort']==c).sum() > 0]
                if subset["cohort"].isna().any():
                    _parts.append(f"unlabeled: {subset['cohort'].isna().sum()}")
                tit = f"{grp}\n" + "  ".join(_parts)
            else:
                for age in ["young", "old"]:
                    sub = subset[subset["age_group"] == age]
                    if len(sub) == 0:
                        continue
                    for _, r in sub.iterrows():
                        pts = df_seg[
                            (df_seg["experiment_id"] == r["experiment_id"])
                            & (df_seg["detection_run_id"] == r["detection_run_id"])
                            & (df_seg["seg_40"] == r["seg_40"])
                        ].sort_values(["frame", "time_s"])
                        x = pts["x"].values.astype(float) - pts["x"].iloc[0]
                        y = pts["y"].values.astype(float) - pts["y"].iloc[0]
                        ax.plot(x, y, color="steelblue" if age == "young" else "coral", alpha=0.20, linewidth=0.7)
                n_young = (subset["age_group"] == "young").sum()
                n_old = (subset["age_group"] == "old").sum()
                tit = f"{grp}\nyoung: {n_young}  old: {n_old}"
            if "Curved" in grp:
                tit += f"\navg turn {subset['mean_abs_turn_deg'].mean():.0f}°"
            elif "Zig-zag" in grp:
                tit += f"\n{subset['n_turns'].mean():.1f} turns/seg"
            ax.set_title(tit, fontsize=9)
            ax.set_xlabel("x (px from start)", fontsize=8)
            ax.set_ylabel("y (px from start)", fontsize=8)
            ax.axis("equal")

        for j in range(n_groups, axes.size):
            axes.flat[j].set_visible(False)

        _leg = (
            "YM / YF / OM / OF = blue / green / orange / red"
            if _use_four_cohorts else "blue=young  coral=old"
        )
        plt.suptitle(
            f"Step 0 — movement trajectories per pattern  ({seg_mm} mm)  {_leg}",
            y=1.02, fontsize=11
        )
        plt.tight_layout()
        save_fig("step0_movements_per_pattern.png")
        plt.close()

        if _use_four_cohorts:
            _bp = seg_clean.dropna(subset=["cohort"]).copy()
            _bp["cohort"] = pd.Categorical(_bp["cohort"], categories=_COHORT_ORDER, ordered=True)
            _pal_b = {k: _COHORT_PALETTE[k] for k in _COHORT_ORDER}
            fig_b1, ax_b1 = plt.subplots(figsize=(12, 5))
            sns.boxplot(
                data=_bp, x="pattern_group", y="mean_abs_turn_deg",
                hue="cohort", palette=_pal_b, ax=ax_b1,
            )
            ax_b1.set_title("Avg turn angle per pattern")
            ax_b1.set_xticklabels(ax_b1.get_xticklabels(), rotation=20, ha="right", fontsize=8)
            ax_b1.legend(bbox_to_anchor=(1.02, 1), loc="upper left", borderaxespad=0, title="Cohort")
            fig_b1.tight_layout(rect=[0, 0, 0.78, 1])
            save_fig("step0_movements_avg_turn_per_pattern.png", fig_b1)
            plt.close(fig_b1)

            fig_b2, ax_b2 = plt.subplots(figsize=(12, 5))
            sns.boxplot(
                data=_bp, x="pattern_group", y="n_turns",
                hue="cohort", palette=_pal_b, ax=ax_b2,
            )
            ax_b2.set_title("Number of turns per pattern")
            ax_b2.set_xticklabels(ax_b2.get_xticklabels(), rotation=20, ha="right", fontsize=8)
            ax_b2.legend(bbox_to_anchor=(1.02, 1), loc="upper left", borderaxespad=0, title="Cohort")
            fig_b2.tight_layout(rect=[0, 0, 0.78, 1])
            save_fig("step0_movements_n_turns_per_pattern.png", fig_b2)
            plt.close(fig_b2)
        else:
            _age_palette = {"young": "steelblue", "old": "coral"}
            fig_b1, ax_b1 = plt.subplots(figsize=(10, 5))
            sns.boxplot(
                data=seg_clean, x="pattern_group", y="mean_abs_turn_deg",
                hue="age_group", palette=_age_palette, ax=ax_b1,
            )
            ax_b1.set_title("Avg turn angle per pattern")
            ax_b1.set_xticklabels(ax_b1.get_xticklabels(), rotation=20, ha="right", fontsize=8)
            ax_b1.legend(bbox_to_anchor=(1.02, 1), loc="upper left", borderaxespad=0, title="Age group")
            fig_b1.tight_layout(rect=[0, 0, 0.78, 1])
            save_fig("step0_movements_avg_turn_per_pattern.png", fig_b1)
            plt.close(fig_b1)

            fig_b2, ax_b2 = plt.subplots(figsize=(10, 5))
            sns.boxplot(
                data=seg_clean, x="pattern_group", y="n_turns",
                hue="age_group", palette=_age_palette, ax=ax_b2,
            )
            ax_b2.set_title("Number of turns per pattern")
            ax_b2.set_xticklabels(ax_b2.get_xticklabels(), rotation=20, ha="right", fontsize=8)
            ax_b2.legend(bbox_to_anchor=(1.02, 1), loc="upper left", borderaxespad=0, title="Age group")
            fig_b2.tight_layout(rect=[0, 0, 0.78, 1])
            save_fig("step0_movements_n_turns_per_pattern.png", fig_b2)
            plt.close(fig_b2)

        fig_s, ax_s = plt.subplots(figsize=(10, 6.5))
        sns.scatterplot(
            data=seg_clean, x="mean_abs_turn_deg", y="n_turns",
            hue="pattern_group", alpha=0.35, s=10, ax=ax_s,
        )
        ax_s.set_title("Turn angle vs n_turns")
        ax_s.legend(
            bbox_to_anchor=(0.5, -0.14),
            loc="upper center",
            ncol=3,
            fontsize=8,
            title="Pattern",
            title_fontsize=9,
            frameon=True,
            columnspacing=0.9,
            handletextpad=0.4,
        )
        fig_s.tight_layout()
        fig_s.subplots_adjust(bottom=0.24)
        save_fig("step0_movements_turn_angle_vs_n_turns.png", fig_s)
        plt.close(fig_s)

        # ── Plot 5: actual coordinates — fly movement in the vial (ALL trajectories) ──
        n_cols5 = min(3, n_groups)
        n_rows5 = int(np.ceil(n_groups / n_cols5))
        fig5, axes5 = plt.subplots(n_rows5, n_cols5, figsize=(6 * n_cols5, 5 * n_rows5))
        axes5 = np.atleast_2d(axes5)

        for idx_g, grp in enumerate(groups_sorted):
            ax5 = axes5.flat[idx_g]
            subset = seg_clean[seg_clean["pattern_group"] == grp]
            if _use_four_cohorts:
                for c in _COHORT_ORDER:
                    sub = subset[subset["cohort"] == c]
                    if len(sub) == 0:
                        continue
                    for _, r in sub.iterrows():
                        pts = df_seg[
                            (df_seg["experiment_id"] == r["experiment_id"])
                            & (df_seg["detection_run_id"] == r["detection_run_id"])
                            & (df_seg["seg_40"] == r["seg_40"])
                        ].sort_values(["frame", "time_s"])
                        ax5.plot(
                            pts["x"].values,
                            pts["y"].values,
                            color=_COHORT_PALETTE[c],
                            alpha=0.30,
                            linewidth=0.8,
                        )
                for _, r in subset[subset["cohort"].isna()].iterrows():
                    pts = df_seg[
                        (df_seg["experiment_id"] == r["experiment_id"])
                        & (df_seg["detection_run_id"] == r["detection_run_id"])
                        & (df_seg["seg_40"] == r["seg_40"])
                    ].sort_values(["frame", "time_s"])
                    ax5.plot(
                        pts["x"].values,
                        pts["y"].values,
                        color="#999999",
                        alpha=0.15,
                        linewidth=0.7,
                    )
                _parts5 = [f"{c}: {(subset['cohort']==c).sum()}" for c in _COHORT_ORDER if (subset['cohort']==c).sum() > 0]
                if subset["cohort"].isna().any():
                    _parts5.append(f"unlabeled: {subset['cohort'].isna().sum()}")
                tit = f"{grp}\n" + "  ".join(_parts5)
            else:
                for age in ["young", "old"]:
                    sub = subset[subset["age_group"] == age]
                    if len(sub) == 0:
                        continue
                    for _, r in sub.iterrows():
                        pts = df_seg[
                            (df_seg["experiment_id"] == r["experiment_id"])
                            & (df_seg["detection_run_id"] == r["detection_run_id"])
                            & (df_seg["seg_40"] == r["seg_40"])
                        ].sort_values(["frame", "time_s"])
                        ax5.plot(
                            pts["x"].values,
                            pts["y"].values,
                            color="steelblue" if age == "young" else "coral",
                            alpha=0.30,
                            linewidth=0.8,
                        )
                n_young = (subset["age_group"] == "young").sum()
                n_old = (subset["age_group"] == "old").sum()
                tit = f"{grp}\nyoung: {n_young}  old: {n_old}"
            ax5.set_title(tit, fontsize=9)
            ax5.set_xlabel("x (px)", fontsize=8)
            ax5.set_ylabel("y (px)", fontsize=8)
            ax5.invert_yaxis()   # image-space: y=0 at top
            ax5.set_aspect("equal", adjustable="datalim")

        for j in range(n_groups, axes5.size):
            axes5.flat[j].set_visible(False)

        plt.suptitle(
            f"Step 0 — actual vial positions per pattern  ({seg_mm} mm)  {_leg}",
            y=1.02, fontsize=11
        )
        plt.tight_layout()
        save_fig("step0_actual_positions.png")
        plt.close()

    globals()["PLOT_DIR"] = old_plot_dir

sum_df = pd.DataFrame(summary_rows)
sum_path = BASE0 / "step0_pattern_summary.csv"
sum_df.to_csv(sum_path, index=False)
print(f"\nStep 0 summary saved: {sum_path.resolve()}")
display(sum_df)
print("Done. Step 0 (global pooled merged CSV).")           



  Step 0 5mm: n_segments=31,664  ->  k=6 patterns  (distinct labels: 6)  silhouette=0.3745
  Step 0 10mm: n_segments=32,178  ->  k=3 patterns  (distinct labels: 3)  silhouette=0.3421
  Step 0 15mm: n_segments=22,132  ->  k=4 patterns  (distinct labels: 4)  silhouette=0.3318
  Step 0 20mm: n_segments=16,868  ->  k=4 patterns  (distinct labels: 4)  silhouette=0.3133
  Step 0 30mm: n_segments=11,395  ->  k=6 patterns  (distinct labels: 6)  silhouette=0.2756
  Step 0 40mm: n_segments=8,591  ->  k=5 patterns  (distinct labels: 5)  silhouette=0.2617

Step 0 summary saved: /Users/yashpatel/Documents/John-Tower-Lab/paper_trajectory/Exp1/pattern_distance_output/0_global/step0_pattern_summary.csv


,segment_length_mm,n_segments,n_patterns_k,n_distinct_labels,best_silhouette,labels
0,5,31664,6,6,0.374532,curved; direct; meandering; straight; winding;...
1,10,32178,3,3,0.342117,meandering; straight; zig-zag
2,15,22132,4,4,0.331805,curved; meandering; straight; zig-zag
3,20,16868,4,4,0.313312,curved; meandering; straight; zig-zag
4,30,11395,6,6,0.275605,curved; direct; meandering; straight; winding;...
5,40,8591,5,5,0.261706,curved; meandering; straight; winding; zig-zag


Done. Step 0 (global pooled merged CSV).


In [6]:
# Step 1: one row per original detection CSV = one (experiment_id, detection_run_id).
# For EACH fly/CSV, for EACH segment length folder (5mm, 10mm, …), we write EXACTLY 5 PNG files
# (same types as Step 0): step1_silhouette_k, step1_pattern_counts, step1_pca_patterns,
# step1_movements_per_pattern, step1_actual_positions. Plus step1_pattern_summary.csv once per run.
DISTANCES_MM = [5, 10, 15, 20, 30, 40]
MIN_PATH_MM = 5.0
BASE_DIR = Path("pattern_distance_output") / "1_entire_data"
run_list = df[["experiment_id", "detection_run_id"]].drop_duplicates().apply(tuple, axis=1).tolist()

for (exp_id, run_id) in run_list:
    df_sub = df[(df["experiment_id"] == exp_id) & (df["detection_run_id"] == run_id)].copy()
    if len(df_sub) == 0:
        print(f"  Skip 1_entire_data {exp_id} run_{run_id}: no data")
        continue
    run_label = f"{exp_id}_run_{run_id}"
    label = f"Step 1 — {run_label}"
    summary_rows = []

    for seg_mm in DISTANCES_MM:
        out_dir = BASE_DIR / run_label / f"{seg_mm}mm"
        out_dir.mkdir(parents=True, exist_ok=True)
        old_plot_dir = PLOT_DIR
        globals()["PLOT_DIR"] = out_dir
        SEGMENT_LENGTH_MM = seg_mm

        df_seg = df_sub.dropna(subset=["cum_dist_mm", "total_dist_mm"]).copy()
        df_seg = df_seg[df_seg["total_dist_mm"] > 0].copy()
        df_seg["seg_40"] = (df_seg["cum_dist_mm"] // SEGMENT_LENGTH_MM).astype(int) + 1
        n_segs_per_video = (
            df_seg.groupby(video_cols)["total_dist_mm"].transform("first") / SEGMENT_LENGTH_MM
        ).apply(np.ceil).astype(int)
        df_seg["seg_40"] = df_seg["seg_40"].clip(upper=n_segs_per_video)

        seg = (
            df_seg.groupby(video_cols + ["category", "age_group", "seg_40"], as_index=False)
            .agg(
                n_rows=("frame", "size"),
                t_start_s=("time_s", "min"),
                t_end_s=("time_s", "max"),
                seg_path_mm=("step_dist_mm", "sum"),
                speed_mean_mm_s=("speed_mm_s", "mean"),
                x_start=("x", "first"),
                y_start=("y", "first"),
                x_end=("x", "last"),
                y_end=("y", "last"),
            )
        )
        seg["seg_dur_s"] = seg["t_end_s"] - seg["t_start_s"]
        seg["net_dx_px"] = seg["x_end"] - seg["x_start"]
        seg["net_dy_px"] = seg["y_end"] - seg["y_start"]
        seg["net_disp_px"] = np.sqrt(seg["net_dx_px"] ** 2 + seg["net_dy_px"] ** 2)
        cal = df_seg.groupby(video_cols, as_index=False).agg(
            mm_per_px_x=("mm_per_px_x", "first"),
            mm_per_px_y=("mm_per_px_y", "first"),
        )
        seg = seg.merge(cal, on=video_cols, how="left")
        seg["net_disp_mm"] = np.sqrt(
            (seg["net_dx_px"] * seg["mm_per_px_x"]) ** 2 + (seg["net_dy_px"] * seg["mm_per_px_y"]) ** 2
        )
        seg["straightness"] = seg["net_disp_mm"] / seg["seg_path_mm"].replace({0: np.nan})
        seg = seg[seg["seg_path_mm"] >= MIN_PATH_MM].copy()
        seg["tortuosity"] = seg["seg_path_mm"] / seg["net_disp_mm"].replace(0, np.nan).clip(lower=1e-6)
        seg["path_per_40"] = seg["seg_path_mm"] / SEGMENT_LENGTH_MM

        mean_turns, n_turns_list = [], []
        for _, r in seg.iterrows():
            sub = df_seg[
                (df_seg["experiment_id"] == r["experiment_id"])
                & (df_seg["detection_run_id"] == r["detection_run_id"])
                & (df_seg["seg_40"] == r["seg_40"])
            ].sort_values(["frame", "time_s"])
            ma, nt = turn_stats_from_xy(sub["x"].values, sub["y"].values)
            mean_turns.append(ma)
            n_turns_list.append(nt)
        seg["mean_abs_turn_deg"] = mean_turns
        seg["n_turns"] = n_turns_list
        seg = seg.dropna(subset=["straightness", "tortuosity", "path_per_40"]).copy()
        seg["mean_abs_turn_deg"] = seg["mean_abs_turn_deg"].fillna(0)

        if len(seg) < 10:
            print(f"  {label} {seg_mm}mm: too few segments ({len(seg)}), skip")
            globals()["PLOT_DIR"] = old_plot_dir
            continue

        X = seg[FEATURE_COLS].copy()
        _feat_scaler = StandardScaler()
        X_scaled = _feat_scaler.fit_transform(X)
        K_RANGE = range(2, max(3, min(11, len(seg) // 5)))
        silhouettes, inertias = [], []
        for k in K_RANGE:
            km_trial = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X_scaled)
            silhouettes.append(silhouette_score(X_scaled, km_trial.predict(X_scaled)))
            inertias.append(km_trial.inertia_)
        best_k = list(K_RANGE)[np.argmax(silhouettes)] if silhouettes else 2
        n_clusters = best_k
        km = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
        seg["pattern"] = km.fit_predict(X_scaled)
        seg["pattern"] = seg["pattern"].astype(int)
        seg_clean = seg.copy()

        means = seg_clean.groupby("pattern")[FEATURE_COLS].mean()
        patterns = sorted(means.index.tolist())
        assigned = set()

        def assign_best(lbl, feature, order="max"):
            remaining = [p for p in patterns if p not in assigned]
            if not remaining:
                return None
            pat = (
                means.loc[remaining, feature].idxmax()
                if order == "max"
                else means.loc[remaining, feature].idxmin()
            )
            assigned.add(pat)
            return pat

        label_specs = [
            ("zig-zag", "n_turns", "max"),
            ("straight", "straightness", "max"),
            ("meandering", "path_per_40", "max"),
            ("curved", "mean_abs_turn_deg", "max"),
            ("winding", "tortuosity", "max"),
            ("direct", "straightness", "max"),
            ("exploratory", "path_per_40", "max"),
        ]
        pattern_labels = {}
        for lab, feat, ord_ in label_specs[:n_clusters]:
            p = assign_best(lab, feat, ord_)
            if p is not None:
                pattern_labels[p] = lab
        for p in patterns:
            if p not in pattern_labels:
                pattern_labels[p] = "moderate"
        seg_clean["pattern_label"] = seg_clean["pattern"].map(pattern_labels)
        PATTERN_GROUP = {
            "straight": "Straight",
            "direct": "Direct",
            "meandering": "Meandering",
            "zig-zag": "Zig-zag / reversing",
            "curved": "Curved / turning",
            "winding": "Winding",
            "exploratory": "Exploratory",
            "moderate": "Other / moderate",
        }
        seg_clean["pattern_group"] = seg_clean["pattern_label"].map(PATTERN_GROUP)

        distinct_names = sorted(set(seg_clean["pattern_label"].astype(str)))
        best_sil = float(max(silhouettes)) if silhouettes else float("nan")
        summary_rows.append(
            {
                "experiment_id": exp_id,
                "detection_run_id": run_id,
                "segment_length_mm": seg_mm,
                "n_segments": len(seg_clean),
                "n_patterns_k": int(n_clusters),
                "n_distinct_labels": len(distinct_names),
                "best_silhouette": best_sil,
                "labels": "; ".join(distinct_names),
            }
        )
        print(
            f"  {label} {seg_mm}mm: n_segments={len(seg_clean):,}  ->  k={n_clusters}  "
            f"silhouette={best_sil:.4f}"
        )

        export_pattern_feature_stats_csv(
            seg_clean,
            FEATURE_COLS,
            out_dir,
            segment_length_mm=seg_mm,
            step_name="step1_per_fly",
            experiment_id=exp_id,
            detection_run_id=run_id,
            only_cluster_naming_order_csv=ONLY_CLUSTER_NAMING_ORDER_CSV,
        )

        _pca = PCA(n_components=2, random_state=42)
        pc_xy = _pca.fit_transform(X_scaled)
        export_pca_specification_csv(
            _feat_scaler,
            _pca,
            FEATURE_COLS,
            out_dir,
            analysis_id=run_label,
            segment_length_mm=seg_mm,
            n_segments=len(seg_clean),
            k_patterns=int(n_clusters),
            experiment_id=str(exp_id),
            detection_run_id=int(run_id),
        )

        if not SKIP_PATTERN_FIGURES:
            # ── Plot 1: silhouette + inertia (same as Step 0) ───────────────────────
            fig, ax = plt.subplots(1, 2, figsize=(10, 4))
            ax[0].plot(list(K_RANGE), silhouettes, "o-")
            ax[0].axvline(
                n_clusters, color="green", linestyle="--", alpha=0.8, label=f"chosen k={n_clusters}"
            )
            ax[0].set_xlabel("k (number of patterns)")
            ax[0].set_ylabel("Silhouette score")
            ax[0].legend()
            ax[1].plot(list(K_RANGE), inertias, "o-")
            ax[1].axvline(n_clusters, color="green", linestyle="--", alpha=0.8)
            ax[1].set_xlabel("k")
            ax[1].set_ylabel("Inertia")
            plt.suptitle(f"{label} — {seg_mm} mm segments", y=1.02)
            plt.tight_layout()
            save_fig("step1_silhouette_k.png")
            plt.close()

            # ── Plot 2: segments per pattern (bar, same as Step 0) ────────────────────
            fig, ax = plt.subplots(figsize=(9, 4))
            seg_clean["pattern_label"].value_counts().sort_index().plot(
                kind="bar", ax=ax, color="steelblue", edgecolor="gray"
            )
            ax.set_ylabel("Segment count")
            ax.set_xlabel("Pattern label")
            ax.set_title(f"{label}: segments per pattern (k={n_clusters}, {seg_mm} mm)")
            plt.xticks(rotation=25, ha="right")
            plt.tight_layout()
            save_fig("step1_pattern_counts.png")
            plt.close()

            # ── Plot 3: PCA by pattern (same as Step 0) ───────────────────────────────
            seg_clean["PC1"] = pc_xy[:, 0]
            seg_clean["PC2"] = pc_xy[:, 1]
            fig, ax = plt.subplots(figsize=(10, 6.5))
            sns.scatterplot(
                data=seg_clean, x="PC1", y="PC2", hue="pattern_label", alpha=0.25, s=8, ax=ax
            )
            ax.set_title(f"{label} — PCA ({seg_mm} mm, k={n_clusters})")
            ax.legend(
                bbox_to_anchor=(0.5, -0.14),
                loc="upper center",
                ncol=3,
                fontsize=8,
                title="Pattern",
                title_fontsize=9,
                frameon=True,
                columnspacing=0.9,
                handletextpad=0.4,
            )
            fig.tight_layout()
            fig.subplots_adjust(bottom=0.24)
            save_fig("step1_pca_patterns.png")
            plt.close()

            # ── Plot 4: centered trajectories + turn stats (same layout as Step 0) ───
            groups_sorted = seg_clean["pattern_group"].dropna().unique()
            groups_sorted = sorted(groups_sorted, key=lambda g: -(seg_clean["pattern_group"] == g).sum())
            n_groups = len(groups_sorted)
            n_cols = min(3, max(1, n_groups))
            n_rows = int(np.ceil(n_groups / n_cols)) if n_groups else 1
            fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 5 * n_rows))
            axes = np.atleast_2d(axes)

            for idx_g, grp in enumerate(groups_sorted):
                axp = axes.flat[idx_g]
                subset = seg_clean[seg_clean["pattern_group"] == grp]
                for age in ["young", "old"]:
                    sub = subset[subset["age_group"] == age]
                    if len(sub) == 0:
                        continue
                    for _, r in sub.iterrows():
                        pts = df_seg[
                            (df_seg["experiment_id"] == r["experiment_id"])
                            & (df_seg["detection_run_id"] == r["detection_run_id"])
                            & (df_seg["seg_40"] == r["seg_40"])
                        ].sort_values(["frame", "time_s"])
                        x = pts["x"].values.astype(float) - pts["x"].iloc[0]
                        y = pts["y"].values.astype(float) - pts["y"].iloc[0]
                        axp.plot(
                            x,
                            y,
                            color="steelblue" if age == "young" else "coral",
                            alpha=0.20,
                            linewidth=0.7,
                        )
                n_young = (subset["age_group"] == "young").sum()
                n_old = (subset["age_group"] == "old").sum()
                tit = f"{grp}\nyoung: {n_young}  old: {n_old}"
                if "Curved" in grp:
                    tit += f"\navg turn {subset['mean_abs_turn_deg'].mean():.0f}°"
                elif "Zig-zag" in grp:
                    tit += f"\n{subset['n_turns'].mean():.1f} turns/seg"
                axp.set_title(tit, fontsize=9)
                axp.set_xlabel("x (px from start)", fontsize=8)
                axp.set_ylabel("y (px from start)", fontsize=8)
                axp.axis("equal")

            for j in range(n_groups, axes.size):
                axes.flat[j].set_visible(False)

            plt.suptitle(
                f"{label} — movement trajectories per pattern ({seg_mm} mm)  blue=young  coral=old",
                y=1.02,
                fontsize=11,
            )
            plt.tight_layout()
            save_fig("step1_movements_per_pattern.png")
            plt.close()

            _age_palette = {"young": "steelblue", "old": "coral"}
            fig_b1, ax_b1 = plt.subplots(figsize=(10, 5))
            sns.boxplot(
                data=seg_clean,
                x="pattern_group",
                y="mean_abs_turn_deg",
                hue="age_group",
                palette=_age_palette,
                ax=ax_b1,
            )
            ax_b1.set_title("Avg turn angle per pattern")
            ax_b1.set_xticklabels(ax_b1.get_xticklabels(), rotation=20, ha="right", fontsize=8)
            ax_b1.legend(bbox_to_anchor=(1.02, 1), loc="upper left", borderaxespad=0, title="Age group")
            fig_b1.tight_layout(rect=[0, 0, 0.78, 1])
            save_fig("step1_movements_avg_turn_per_pattern.png", fig_b1)
            plt.close(fig_b1)

            fig_b2, ax_b2 = plt.subplots(figsize=(10, 5))
            sns.boxplot(
                data=seg_clean,
                x="pattern_group",
                y="n_turns",
                hue="age_group",
                palette=_age_palette,
                ax=ax_b2,
            )
            ax_b2.set_title("Number of turns per pattern")
            ax_b2.set_xticklabels(ax_b2.get_xticklabels(), rotation=20, ha="right", fontsize=8)
            ax_b2.legend(bbox_to_anchor=(1.02, 1), loc="upper left", borderaxespad=0, title="Age group")
            fig_b2.tight_layout(rect=[0, 0, 0.78, 1])
            save_fig("step1_movements_n_turns_per_pattern.png", fig_b2)
            plt.close(fig_b2)

            fig_s, ax_s = plt.subplots(figsize=(10, 6.5))
            sns.scatterplot(
                data=seg_clean,
                x="mean_abs_turn_deg",
                y="n_turns",
                hue="pattern_group",
                alpha=0.35,
                s=10,
                ax=ax_s,
            )
            ax_s.set_title("Turn angle vs n_turns")
            ax_s.legend(
                bbox_to_anchor=(0.5, -0.14),
                loc="upper center",
                ncol=3,
                fontsize=8,
                title="Pattern",
                title_fontsize=9,
                frameon=True,
                columnspacing=0.9,
                handletextpad=0.4,
            )
            fig_s.tight_layout()
            fig_s.subplots_adjust(bottom=0.24)
            save_fig("step1_movements_turn_angle_vs_n_turns.png", fig_s)
            plt.close(fig_s)

            # ── Plot 5: actual vial coordinates (same as Step 0) ──────────────────────
            n_cols5 = min(3, n_groups)
            n_rows5 = int(np.ceil(n_groups / n_cols5))
            fig5, axes5 = plt.subplots(n_rows5, n_cols5, figsize=(6 * n_cols5, 5 * n_rows5))
            axes5 = np.atleast_2d(axes5)

            for idx_g, grp in enumerate(groups_sorted):
                ax5 = axes5.flat[idx_g]
                subset = seg_clean[seg_clean["pattern_group"] == grp]
                for age in ["young", "old"]:
                    sub = subset[subset["age_group"] == age]
                    if len(sub) == 0:
                        continue
                    for _, r in sub.iterrows():
                        pts = df_seg[
                            (df_seg["experiment_id"] == r["experiment_id"])
                            & (df_seg["detection_run_id"] == r["detection_run_id"])
                            & (df_seg["seg_40"] == r["seg_40"])
                        ].sort_values(["frame", "time_s"])
                        ax5.plot(
                            pts["x"].values,
                            pts["y"].values,
                            color="steelblue" if age == "young" else "coral",
                            alpha=0.30,
                            linewidth=0.8,
                        )
                n_young = (subset["age_group"] == "young").sum()
                n_old = (subset["age_group"] == "old").sum()
                tit = f"{grp}\nyoung: {n_young}  old: {n_old}"
                ax5.set_title(tit, fontsize=9)
                ax5.set_xlabel("x (px)", fontsize=8)
                ax5.set_ylabel("y (px)", fontsize=8)
                ax5.invert_yaxis()
                ax5.set_aspect("equal", adjustable="datalim")

            for j in range(n_groups, axes5.size):
                axes5.flat[j].set_visible(False)

            plt.suptitle(
                f"{label} — actual vial positions per pattern ({seg_mm} mm)  blue=young  coral=old",
                y=1.02,
                fontsize=11,
            )
            plt.tight_layout()
            save_fig("step1_actual_positions.png")
            plt.close()

        globals()["PLOT_DIR"] = old_plot_dir
        _step1_pngs = [
            "step1_silhouette_k.png",
            "step1_pattern_counts.png",
            "step1_pca_patterns.png",
            "step1_movements_per_pattern.png",
            "step1_movements_avg_turn_per_pattern.png",
            "step1_movements_n_turns_per_pattern.png",
            "step1_movements_turn_angle_vs_n_turns.png",
            "step1_actual_positions.png",
        ]
        _missing = [n for n in _step1_pngs if not (out_dir / n).is_file()]
        if _missing:
            print(f"  ERROR {run_label} {seg_mm}mm: expected {len(_step1_pngs)} PNGs, missing: {_missing}")
        else:
            print(f"  OK — {len(_step1_pngs)}/{len(_step1_pngs)} PNGs for {run_label} {seg_mm}mm -> {out_dir}")

    sum_df = pd.DataFrame(summary_rows)
    if len(sum_df):
        run_sum_path = BASE_DIR / run_label / "step1_pattern_summary.csv"
        run_sum_path.parent.mkdir(parents=True, exist_ok=True)
        sum_df.to_csv(run_sum_path, index=False)
        print(f"  Step 1 summary CSV: {run_sum_path.resolve()}")

print("Done. Step 1 (per video, same outputs as Step 0).")



  Step 1 — sd_09_12_run_1 5mm: n_segments=671  ->  k=2  silhouette=0.4341
  OK — 8/8 PNGs for sd_09_12_run_1 5mm -> pattern_distance_output/1_entire_data/sd_09_12_run_1/5mm
  Step 1 — sd_09_12_run_1 10mm: n_segments=686  ->  k=2  silhouette=0.3619
  OK — 8/8 PNGs for sd_09_12_run_1 10mm -> pattern_distance_output/1_entire_data/sd_09_12_run_1/10mm
  Step 1 — sd_09_12_run_1 15mm: n_segments=493  ->  k=2  silhouette=0.3197
  OK — 8/8 PNGs for sd_09_12_run_1 15mm -> pattern_distance_output/1_entire_data/sd_09_12_run_1/15mm
  Step 1 — sd_09_12_run_1 20mm: n_segments=385  ->  k=5  silhouette=0.3183
  OK — 8/8 PNGs for sd_09_12_run_1 20mm -> pattern_distance_output/1_entire_data/sd_09_12_run_1/20mm
  Step 1 — sd_09_12_run_1 30mm: n_segments=265  ->  k=7  silhouette=0.2780
  OK — 8/8 PNGs for sd_09_12_run_1 30mm -> pattern_distance_output/1_entire_data/sd_09_12_run_1/30mm
  Step 1 — sd_09_12_run_1 40mm: n_segments=200  ->  k=2  silhouette=0.2779
  OK — 8/8 PNGs for sd_09_12_run_1 40mm -> patte

In [7]:
# Step 2: Males vs females pooled — young vs old by sex (age_group + sex) across full merged CSV
DISTANCES_MM = [5, 10, 15, 20, 30, 40]
MIN_PATH_MM = 5.0

for folder_name, (young_part, old_part) in [
    ("males", (
        df[(df["age_group"] == "young") & (df["sex"] == "male")],
        df[(df["age_group"] == "old") & (df["sex"] == "male")],
    )),
    ("females", (
        df[(df["age_group"] == "young") & (df["sex"] == "female")],
        df[(df["age_group"] == "old") & (df["sex"] == "female")],
    )),
]:
    df_sub = pd.concat([young_part, old_part], ignore_index=True).copy()
    if len(df_sub) == 0:
        print(f"  Skip 2_{folder_name}: no data")
        continue
    label = f"2_{folder_name}"
    for seg_mm in DISTANCES_MM:
        out_dir = Path("pattern_distance_output") / f"2_{folder_name}" / f"{seg_mm}mm"
        out_dir.mkdir(parents=True, exist_ok=True)
        old_plot_dir = PLOT_DIR
        globals()["PLOT_DIR"] = out_dir
        SEGMENT_LENGTH_MM = seg_mm
        df_seg = df_sub.dropna(subset=["cum_dist_mm", "total_dist_mm"]).copy()
        df_seg = df_seg[df_seg["total_dist_mm"] > 0].copy()
        df_seg["seg_40"] = (df_seg["cum_dist_mm"] // SEGMENT_LENGTH_MM).astype(int) + 1
        n_segs_per_video = (df_seg.groupby(video_cols)["total_dist_mm"].transform("first") / SEGMENT_LENGTH_MM).apply(np.ceil).astype(int)
        df_seg["seg_40"] = df_seg["seg_40"].clip(upper=n_segs_per_video)

        seg = (
            df_seg.groupby(video_cols + ["category", "age_group", "seg_40"], as_index=False)
            .agg(
                n_rows=("frame", "size"),
                t_start_s=("time_s", "min"), t_end_s=("time_s", "max"),
                seg_path_mm=("step_dist_mm", "sum"),
                speed_mean_mm_s=("speed_mm_s", "mean"),
                x_start=("x", "first"), y_start=("y", "first"),
                x_end=("x", "last"), y_end=("y", "last"),
            )
        )
        seg["seg_dur_s"] = seg["t_end_s"] - seg["t_start_s"]
        seg["net_dx_px"] = seg["x_end"] - seg["x_start"]
        seg["net_dy_px"] = seg["y_end"] - seg["y_start"]
        seg["net_disp_px"] = np.sqrt(seg["net_dx_px"] ** 2 + seg["net_dy_px"] ** 2)
        cal = df_seg.groupby(video_cols, as_index=False).agg(mm_per_px_x=("mm_per_px_x", "first"), mm_per_px_y=("mm_per_px_y", "first"))
        seg = seg.merge(cal, on=video_cols, how="left")
        seg["net_disp_mm"] = np.sqrt((seg["net_dx_px"] * seg["mm_per_px_x"]) ** 2 + (seg["net_dy_px"] * seg["mm_per_px_y"]) ** 2)
        seg["straightness"] = seg["net_disp_mm"] / seg["seg_path_mm"].replace({0: np.nan})
        seg = seg[seg["seg_path_mm"] >= MIN_PATH_MM].copy()

        seg["tortuosity"] = seg["seg_path_mm"] / seg["net_disp_mm"].replace(0, np.nan).clip(lower=1e-6)
        seg["path_per_40"] = seg["seg_path_mm"] / SEGMENT_LENGTH_MM
        mean_turns, n_turns_list = [], []
        for _, r in seg.iterrows():
            sub = df_seg[(df_seg["experiment_id"] == r["experiment_id"]) & (df_seg["detection_run_id"] == r["detection_run_id"]) & (df_seg["seg_40"] == r["seg_40"])].sort_values(["frame", "time_s"])
            ma, nt = turn_stats_from_xy(sub["x"].values, sub["y"].values)
            mean_turns.append(ma)
            n_turns_list.append(nt)
        seg["mean_abs_turn_deg"] = mean_turns
        seg["n_turns"] = n_turns_list
        seg = seg.dropna(subset=["straightness", "tortuosity", "path_per_40"]).copy()
        seg["mean_abs_turn_deg"] = seg["mean_abs_turn_deg"].fillna(0)

        if len(seg) < 10:
            print(f"  {label} {seg_mm}mm: too few segments ({len(seg)}), skip")
            globals()["PLOT_DIR"] = old_plot_dir
            continue

        X = seg[FEATURE_COLS].copy()
        _feat_scaler = StandardScaler()
        X_scaled = _feat_scaler.fit_transform(X)
        K_RANGE = range(2, max(3, min(11, len(seg) // 5)))
        silhouettes, inertias = [], []
        for k in K_RANGE:
            km_trial = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X_scaled)
            silhouettes.append(silhouette_score(X_scaled, km_trial.predict(X_scaled)))
            inertias.append(km_trial.inertia_)
        best_k = list(K_RANGE)[np.argmax(silhouettes)] if silhouettes else 2
        n_clusters = best_k
        km = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
        seg["pattern"] = km.fit_predict(X_scaled)
        seg["pattern"] = seg["pattern"].astype(int)
        seg_clean = seg.copy()
        means = seg_clean.groupby("pattern")[FEATURE_COLS].mean()
        patterns = sorted(means.index.tolist())
        assigned = set()
        def assign_best(lbl, feature, order="max"):
            remaining = [p for p in patterns if p not in assigned]
            if not remaining: return None
            pat = means.loc[remaining, feature].idxmax() if order == "max" else means.loc[remaining, feature].idxmin()
            assigned.add(pat)
            return pat
        label_specs = [("zig-zag", "n_turns", "max"), ("straight", "straightness", "max"), ("meandering", "path_per_40", "max"), ("curved", "mean_abs_turn_deg", "max"), ("winding", "tortuosity", "max"), ("direct", "straightness", "max"), ("exploratory", "path_per_40", "max")]
        pattern_labels = {}
        for lab, feat, ord in label_specs[:n_clusters]:
            p = assign_best(lab, feat, ord)
            if p is not None: pattern_labels[p] = lab
        for p in patterns:
            if p not in pattern_labels: pattern_labels[p] = "moderate"
        seg_clean["pattern_label"] = seg_clean["pattern"].map(pattern_labels)
        PATTERN_GROUP = {"straight": "Straight", "direct": "Direct", "meandering": "Meandering", "zig-zag": "Zig-zag / reversing", "curved": "Curved / turning", "winding": "Winding", "exploratory": "Exploratory", "moderate": "Other / moderate"}
        seg_clean["pattern_group"] = seg_clean["pattern_label"].map(PATTERN_GROUP)

        export_pattern_feature_stats_csv(
            seg_clean,
            FEATURE_COLS,
            out_dir,
            segment_length_mm=seg_mm,
            step_name=f"step2_{folder_name}",
            only_cluster_naming_order_csv=ONLY_CLUSTER_NAMING_ORDER_CSV,
        )

        _pca = PCA(n_components=2, random_state=42)
        pc_xy = _pca.fit_transform(X_scaled)
        export_pca_specification_csv(
            _feat_scaler,
            _pca,
            FEATURE_COLS,
            out_dir,
            analysis_id=label,
            segment_length_mm=seg_mm,
            n_segments=len(seg_clean),
            k_patterns=int(n_clusters),
        )

        if not SKIP_PATTERN_FIGURES:
            fig, ax = plt.subplots(1, 2, figsize=(10, 4))
            ax[0].plot(list(K_RANGE), silhouettes, "o-")
            ax[0].axvline(n_clusters, color="green", linestyle="--", alpha=0.8, label=f"used k={n_clusters}")
            ax[0].set_xlabel("Number of clusters (k)")
            ax[0].set_ylabel("Silhouette score")
            ax[0].legend()
            ax[1].plot(list(K_RANGE), inertias, "o-")
            ax[1].axvline(n_clusters, color="green", linestyle="--", alpha=0.8, label=f"used k={n_clusters}")
            ax[1].set_xlabel("Number of clusters (k)")
            ax[1].set_ylabel("Inertia")
            ax[1].legend()
            plt.tight_layout()
            save_fig("pattern_k_selection.png")
            plt.close()

            fig, ax = plt.subplots(figsize=(10, 5))
            pt = seg_clean.groupby(["pattern", "age_group"]).size().unstack(fill_value=0).reindex(sorted(seg_clean["pattern"].unique()))
            pt.index = [f"{i} ({pattern_labels.get(i, '?')})" for i in pt.index]
            pt.plot(kind="bar", stacked=True, ax=ax, color=["steelblue", "coral"], edgecolor="gray")
            ax.set_ylabel(f"Number of {seg_mm}mm segments")
            ax.set_title(f"{label}: pattern by age group")
            ax.legend(title="Age group")
            ax.set_xticklabels(ax.get_xticklabels(), rotation=15, ha="right")
            plt.tight_layout()
            save_fig("pattern_by_age_group_stacked.png")
            plt.close()

            fig, axes = plt.subplots(1, 2, figsize=(12, 5.5))
            group_counts = seg_clean.groupby(["pattern_group", "age_group"]).size().unstack(fill_value=0)
            group_counts.plot(kind="bar", stacked=True, ax=axes[0], color=["steelblue", "coral"])
            axes[0].set_title(f"{label} {seg_mm}mm: by pattern group")
            axes[0].tick_params(axis="x", rotation=15)
            seg_clean["PC1"] = pc_xy[:, 0]
            seg_clean["PC2"] = pc_xy[:, 1]
            sns.scatterplot(data=seg_clean, x="PC1", y="PC2", hue="pattern_label", alpha=0.6, ax=axes[1])
            axes[1].set_title("PCA by pattern")
            _h_pca, _lab_pca = axes[1].get_legend_handles_labels()
            axes[1].get_legend().remove()
            fig.legend(
                _h_pca,
                _lab_pca,
                bbox_to_anchor=(0.5, 0.02),
                loc="upper center",
                ncol=3,
                fontsize=8,
                title="Pattern",
                title_fontsize=9,
                frameon=True,
                columnspacing=0.9,
                handletextpad=0.4,
            )
            fig.tight_layout()
            fig.subplots_adjust(bottom=0.24)
            save_fig("pattern_pca_by_cluster_and_age.png")
            plt.close()

            groups_sorted = seg_clean["pattern_group"].dropna().unique()
            groups_sorted = sorted(groups_sorted, key=lambda g: -(seg_clean["pattern_group"] == g).sum())
            n_groups = len(groups_sorted)
            n_cols = min(3, max(1, n_groups))
            n_rows = int(np.ceil(n_groups / n_cols)) if n_groups else 1
            fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 5 * n_rows))
            axes = np.atleast_2d(axes)
            for idx, grp in enumerate(groups_sorted):
                ax = axes.flat[idx]
                subset = seg_clean[seg_clean["pattern_group"] == grp]
                for age in ["young", "old"]:
                    sub = subset[subset["age_group"] == age]
                    if len(sub) == 0: continue
                    for _, r in sub.iterrows():
                        s = df_seg[(df_seg["experiment_id"] == r["experiment_id"]) & (df_seg["detection_run_id"] == r["detection_run_id"]) & (df_seg["seg_40"] == r["seg_40"])].sort_values(["frame", "time_s"])
                        x = s["x"].values.astype(float) - s["x"].iloc[0]
                        y = s["y"].values.astype(float) - s["y"].iloc[0]
                        ax.plot(x, y, color="steelblue" if age == "young" else "coral", alpha=0.25, linewidth=0.8)
                n_young = (subset["age_group"] == "young").sum()
                n_old = (subset["age_group"] == "old").sum()
                tit = f"{grp} (young: {n_young}, old: {n_old})"
                if "Curved" in grp:
                    tit += f"\navg turn {subset['mean_abs_turn_deg'].mean():.0f}°"
                elif "Zig-zag" in grp:
                    tit += f"\n{subset['n_turns'].mean():.1f} turns/seg"
                ax.set_title(tit)
                ax.set_xlabel("x (px from start)")
                ax.set_ylabel("y (px from start)")
                ax.axis("equal")
            for j in range(n_groups, axes.size):
                axes.flat[j].set_visible(False)
            plt.suptitle(f"{label} {seg_mm}mm (blue=young, coral=old)", y=1.02)
            plt.tight_layout()
            save_fig("all_movements_by_pattern_young_old.png")
            plt.close()

            _age_palette = {"young": "steelblue", "old": "coral"}
            fig_b1, ax_b1 = plt.subplots(figsize=(10, 5))
            sns.boxplot(data=seg_clean, x="pattern_group", y="mean_abs_turn_deg", hue="age_group", palette=_age_palette, ax=ax_b1)
            ax_b1.set_title("Turn angle by pattern (avg °/seg)")
            ax_b1.set_xticklabels(ax_b1.get_xticklabels(), rotation=15, ha="right")
            ax_b1.legend(bbox_to_anchor=(1.02, 1), loc="upper left", borderaxespad=0, title="Age group")
            fig_b1.tight_layout(rect=[0, 0, 0.78, 1])
            save_fig("movements_avg_turn_per_pattern.png", fig_b1)
            plt.close(fig_b1)

            fig_b2, ax_b2 = plt.subplots(figsize=(10, 5))
            sns.boxplot(data=seg_clean, x="pattern_group", y="n_turns", hue="age_group", palette=_age_palette, ax=ax_b2)
            ax_b2.set_title("Number of turns by pattern")
            ax_b2.set_xticklabels(ax_b2.get_xticklabels(), rotation=15, ha="right")
            ax_b2.legend(bbox_to_anchor=(1.02, 1), loc="upper left", borderaxespad=0, title="Age group")
            fig_b2.tight_layout(rect=[0, 0, 0.78, 1])
            save_fig("movements_n_turns_per_pattern.png", fig_b2)
            plt.close(fig_b2)

            fig_s, ax_s = plt.subplots(figsize=(10, 6.5))
            sns.scatterplot(data=seg_clean, x="mean_abs_turn_deg", y="n_turns", hue="pattern_group", alpha=0.4, ax=ax_s)
            ax_s.set_title("Turn angle vs n_turns (by pattern)")
            ax_s.set_xlabel("Mean abs turn (°)")
            ax_s.legend(
                bbox_to_anchor=(0.5, -0.14),
                loc="upper center",
                ncol=3,
                fontsize=8,
                title="Pattern",
                title_fontsize=9,
                frameon=True,
                columnspacing=0.9,
                handletextpad=0.4,
            )
            fig_s.tight_layout()
            fig_s.subplots_adjust(bottom=0.24)
            save_fig("movements_turn_angle_vs_n_turns.png", fig_s)
            plt.close(fig_s)

            # Actual vial coordinates per pattern (same as Step 0 / Step 1)
            n_cols5 = min(3, n_groups)
            n_rows5 = int(np.ceil(n_groups / n_cols5))
            fig5, axes5 = plt.subplots(n_rows5, n_cols5, figsize=(6 * n_cols5, 5 * n_rows5))
            axes5 = np.atleast_2d(axes5)
            for idx_g, grp in enumerate(groups_sorted):
                ax5 = axes5.flat[idx_g]
                subset = seg_clean[seg_clean["pattern_group"] == grp]
                for age in ["young", "old"]:
                    sub = subset[subset["age_group"] == age]
                    if len(sub) == 0:
                        continue
                    for _, r in sub.iterrows():
                        pts = df_seg[
                            (df_seg["experiment_id"] == r["experiment_id"])
                            & (df_seg["detection_run_id"] == r["detection_run_id"])
                            & (df_seg["seg_40"] == r["seg_40"])
                        ].sort_values(["frame", "time_s"])
                        ax5.plot(
                            pts["x"].values,
                            pts["y"].values,
                            color="steelblue" if age == "young" else "coral",
                            alpha=0.30,
                            linewidth=0.8,
                        )
                n_young = (subset["age_group"] == "young").sum()
                n_old = (subset["age_group"] == "old").sum()
                tit = f"{grp}\nyoung: {n_young}  old: {n_old}"
                if "Curved" in grp:
                    tit += f"\navg turn {subset['mean_abs_turn_deg'].mean():.0f}°"
                elif "Zig-zag" in grp:
                    tit += f"\n{subset['n_turns'].mean():.1f} turns/seg"
                ax5.set_title(tit, fontsize=9)
                ax5.set_xlabel("x (px)", fontsize=8)
                ax5.set_ylabel("y (px)", fontsize=8)
                ax5.invert_yaxis()
                ax5.set_aspect("equal", adjustable="datalim")
            for j in range(n_groups, axes5.size):
                axes5.flat[j].set_visible(False)
            plt.suptitle(
                f"{label} — actual vial positions per pattern ({seg_mm} mm)  blue=young  coral=old",
                y=1.02,
                fontsize=11,
            )
            plt.tight_layout()
            save_fig("actual_positions_by_pattern_young_old.png")
            plt.close()

            SIMILARITY_THRESHOLD = 0.4
            panels = []
            for grp in groups_sorted:
                young_sub = seg_clean[(seg_clean["pattern_group"] == grp) & (seg_clean["age_group"] == "young")]
                old_sub = seg_clean[(seg_clean["pattern_group"] == grp) & (seg_clean["age_group"] == "old")]
                n_young, n_old = len(young_sub), len(old_sub)
                if n_young == 0 and n_old == 0: continue
                if n_young == 0:
                    panels.append((grp, "old"))
                elif n_old == 0:
                    panels.append((grp, "young"))
                else:
                    my_t, mo_t = young_sub["mean_abs_turn_deg"].mean(), old_sub["mean_abs_turn_deg"].mean()
                    my_n, mo_n = young_sub["n_turns"].mean(), old_sub["n_turns"].mean()
                    rel_t = abs(my_t - mo_t) / (max(my_t, mo_t) + 1e-6)
                    rel_n = abs(my_n - mo_n) / (max(my_n, mo_n) + 1e-6)
                    if rel_t < SIMILARITY_THRESHOLD and rel_n < SIMILARITY_THRESHOLD:
                        panels.append((grp, "both"))
                    else:
                        panels.append((grp, "young"))
                        panels.append((grp, "old"))
            n_panels = len(panels)
            if n_panels > 0:
                n_cols_c = min(3, n_panels)
                n_rows_c = int(np.ceil(n_panels / n_cols_c))
                fig_c, axes_c = plt.subplots(n_rows_c, n_cols_c, figsize=(5 * n_cols_c, 5 * n_rows_c))
                axes_c = np.atleast_2d(axes_c)
                for idx, (grp, mode) in enumerate(panels):
                    ax = axes_c.flat[idx]
                    subset = seg_clean[seg_clean["pattern_group"] == grp]
                    if mode == "both":
                        for age in ["young", "old"]:
                            sub = subset[subset["age_group"] == age]
                            if len(sub) == 0: continue
                            for _, r in sub.iterrows():
                                s = df_seg[(df_seg["experiment_id"] == r["experiment_id"]) & (df_seg["detection_run_id"] == r["detection_run_id"]) & (df_seg["seg_40"] == r["seg_40"])].sort_values(["frame", "time_s"])
                                x = s["x"].values.astype(float) - s["x"].iloc[0]
                                y = s["y"].values.astype(float) - s["y"].iloc[0]
                                ax.plot(x, y, color="steelblue" if age == "young" else "coral", alpha=0.25, linewidth=0.8)
                        n_young = (subset["age_group"] == "young").sum()
                        n_old = (subset["age_group"] == "old").sum()
                        ax.set_title(f"{grp} (similar)\nyoung: {n_young}, old: {n_old}")
                    else:
                        sub = subset[subset["age_group"] == mode]
                        for _, r in sub.iterrows():
                            s = df_seg[(df_seg["experiment_id"] == r["experiment_id"]) & (df_seg["detection_run_id"] == r["detection_run_id"]) & (df_seg["seg_40"] == r["seg_40"])].sort_values(["frame", "time_s"])
                            x = s["x"].values.astype(float) - s["x"].iloc[0]
                            y = s["y"].values.astype(float) - s["y"].iloc[0]
                            ax.plot(x, y, color="steelblue" if mode == "young" else "coral", alpha=0.25, linewidth=0.8)
                        ax.set_title(f"{grp} ({mode} only, n={len(sub)}")
                    ax.set_xlabel("x (px from start)")
                    ax.set_ylabel("y (px from start)")
                    ax.axis("equal")
                for j in range(n_panels, axes_c.size):
                    axes_c.flat[j].set_visible(False)
                plt.suptitle(f"{label} {seg_mm}mm: comparable (same) vs individual (young/old)", y=1.02)
                plt.tight_layout()
                save_fig("pattern_comparison_young_vs_old.png")
                plt.close()

        globals()["PLOT_DIR"] = old_plot_dir
        print(f"  Saved {label} {seg_mm}mm (k={n_clusters}) -> {out_dir}")
print("Done. Step 2 (males / females, young vs old).")



  Saved 2_males 5mm (k=6) -> pattern_distance_output/2_males/5mm
  Saved 2_males 10mm (k=3) -> pattern_distance_output/2_males/10mm
  Saved 2_males 15mm (k=4) -> pattern_distance_output/2_males/15mm
  Saved 2_males 20mm (k=4) -> pattern_distance_output/2_males/20mm
  Saved 2_males 30mm (k=5) -> pattern_distance_output/2_males/30mm
  Saved 2_males 40mm (k=5) -> pattern_distance_output/2_males/40mm
  Saved 2_females 5mm (k=6) -> pattern_distance_output/2_females/5mm
  Saved 2_females 10mm (k=3) -> pattern_distance_output/2_females/10mm
  Saved 2_females 15mm (k=3) -> pattern_distance_output/2_females/15mm
  Saved 2_females 20mm (k=4) -> pattern_distance_output/2_females/20mm
  Saved 2_females 30mm (k=6) -> pattern_distance_output/2_females/30mm
  Saved 2_females 40mm (k=6) -> pattern_distance_output/2_females/40mm
Done. Step 2 (males / females, young vs old).


In [8]:
# Step 3: Young vs old — male vs female within each age group (pooled merged CSV)
DISTANCES_MM = [5, 10, 15, 20, 30, 40]
MIN_PATH_MM = 5.0

for folder_name, (male_part, female_part) in [
    ("young", (
        df[(df["age_group"] == "young") & (df["sex"] == "male")].copy(),
        df[(df["age_group"] == "young") & (df["sex"] == "female")].copy(),
    )),
    ("old", (
        df[(df["age_group"] == "old") & (df["sex"] == "male")].copy(),
        df[(df["age_group"] == "old") & (df["sex"] == "female")].copy(),
    )),
]:
    if len(male_part) == 0 and len(female_part) == 0:
        print(f"  Skip 3_{folder_name}: no data")
        continue
    df_sub = pd.concat([male_part, female_part], ignore_index=True)
    label = f"3_{folder_name}"
    for seg_mm in DISTANCES_MM:
        out_dir = Path("pattern_distance_output") / f"3_{folder_name}" / f"{seg_mm}mm"
        out_dir.mkdir(parents=True, exist_ok=True)
        old_plot_dir = PLOT_DIR
        globals()["PLOT_DIR"] = out_dir
        SEGMENT_LENGTH_MM = seg_mm
        df_seg = df_sub.dropna(subset=["cum_dist_mm", "total_dist_mm"]).copy()
        df_seg = df_seg[df_seg["total_dist_mm"] > 0].copy()
        df_seg["seg_40"] = (df_seg["cum_dist_mm"] // SEGMENT_LENGTH_MM).astype(int) + 1
        n_segs_per_video = (df_seg.groupby(video_cols)["total_dist_mm"].transform("first") / SEGMENT_LENGTH_MM).apply(np.ceil).astype(int)
        df_seg["seg_40"] = df_seg["seg_40"].clip(upper=n_segs_per_video)
        seg = (
            df_seg.groupby(video_cols + ["category", "age_group", "sex", "seg_40"], as_index=False)
            .agg(
                n_rows=("frame", "size"),
                t_start_s=("time_s", "min"), t_end_s=("time_s", "max"),
                seg_path_mm=("step_dist_mm", "sum"),
                speed_mean_mm_s=("speed_mm_s", "mean"),
                x_start=("x", "first"), y_start=("y", "first"),
                x_end=("x", "last"), y_end=("y", "last"),
            )
        )
        seg["seg_dur_s"] = seg["t_end_s"] - seg["t_start_s"]
        seg["net_dx_px"] = seg["x_end"] - seg["x_start"]
        seg["net_dy_px"] = seg["y_end"] - seg["y_start"]
        seg["net_disp_px"] = np.sqrt(seg["net_dx_px"] ** 2 + seg["net_dy_px"] ** 2)
        cal = df_seg.groupby(video_cols, as_index=False).agg(mm_per_px_x=("mm_per_px_x", "first"), mm_per_px_y=("mm_per_px_y", "first"))
        seg = seg.merge(cal, on=video_cols, how="left")
        seg["net_disp_mm"] = np.sqrt((seg["net_dx_px"] * seg["mm_per_px_x"]) ** 2 + (seg["net_dy_px"] * seg["mm_per_px_y"]) ** 2)
        seg["straightness"] = seg["net_disp_mm"] / seg["seg_path_mm"].replace({0: np.nan})
        seg = seg[seg["seg_path_mm"] >= MIN_PATH_MM].copy()
        seg["tortuosity"] = seg["seg_path_mm"] / seg["net_disp_mm"].replace(0, np.nan).clip(lower=1e-6)
        seg["path_per_40"] = seg["seg_path_mm"] / SEGMENT_LENGTH_MM
        mean_turns, n_turns_list = [], []
        for _, r in seg.iterrows():
            sub = df_seg[(df_seg["experiment_id"] == r["experiment_id"]) & (df_seg["detection_run_id"] == r["detection_run_id"]) & (df_seg["seg_40"] == r["seg_40"])].sort_values(["frame", "time_s"])
            ma, nt = turn_stats_from_xy(sub["x"].values, sub["y"].values)
            mean_turns.append(ma)
            n_turns_list.append(nt)
        seg["mean_abs_turn_deg"] = mean_turns
        seg["n_turns"] = n_turns_list
        seg = seg.dropna(subset=["straightness", "tortuosity", "path_per_40"]).copy()
        seg["mean_abs_turn_deg"] = seg["mean_abs_turn_deg"].fillna(0)
        if len(seg) < 10:
            print(f"  {label} {seg_mm}mm: too few segments ({len(seg)}), skip")
            globals()["PLOT_DIR"] = old_plot_dir
            continue
        X = seg[FEATURE_COLS].copy()
        _feat_scaler = StandardScaler()
        X_scaled = _feat_scaler.fit_transform(X)
        K_RANGE = range(2, max(3, min(11, len(seg) // 5)))
        silhouettes, inertias = [], []
        for k in K_RANGE:
            km_trial = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X_scaled)
            silhouettes.append(silhouette_score(X_scaled, km_trial.predict(X_scaled)))
            inertias.append(km_trial.inertia_)
        best_k = list(K_RANGE)[np.argmax(silhouettes)] if silhouettes else 2
        n_clusters = best_k
        km = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
        seg["pattern"] = km.fit_predict(X_scaled)
        seg["pattern"] = seg["pattern"].astype(int)
        seg_clean = seg.copy()
        means = seg_clean.groupby("pattern")[FEATURE_COLS].mean()
        patterns = sorted(means.index.tolist())
        assigned = set()
        def assign_best(lbl, feature, order="max"):
            remaining = [p for p in patterns if p not in assigned]
            if not remaining: return None
            pat = means.loc[remaining, feature].idxmax() if order == "max" else means.loc[remaining, feature].idxmin()
            assigned.add(pat)
            return pat
        label_specs = [("zig-zag", "n_turns", "max"), ("straight", "straightness", "max"), ("meandering", "path_per_40", "max"), ("curved", "mean_abs_turn_deg", "max"), ("winding", "tortuosity", "max"), ("direct", "straightness", "max"), ("exploratory", "path_per_40", "max")]
        pattern_labels = {}
        for lab, feat, ord in label_specs[:n_clusters]:
            p = assign_best(lab, feat, ord)
            if p is not None: pattern_labels[p] = lab
        for p in patterns:
            if p not in pattern_labels: pattern_labels[p] = "moderate"
        seg_clean["pattern_label"] = seg_clean["pattern"].map(pattern_labels)
        PATTERN_GROUP = {"straight": "Straight", "direct": "Direct", "meandering": "Meandering", "zig-zag": "Zig-zag / reversing", "curved": "Curved / turning", "winding": "Winding", "exploratory": "Exploratory", "moderate": "Other / moderate"}
        seg_clean["pattern_group"] = seg_clean["pattern_label"].map(PATTERN_GROUP)
        export_pattern_feature_stats_csv(
            seg_clean,
            FEATURE_COLS,
            out_dir,
            segment_length_mm=seg_mm,
            step_name=f"step3_{folder_name}",
            only_cluster_naming_order_csv=ONLY_CLUSTER_NAMING_ORDER_CSV,
        )

        _pca = PCA(n_components=2, random_state=42)
        pc_xy = _pca.fit_transform(X_scaled)
        export_pca_specification_csv(
            _feat_scaler,
            _pca,
            FEATURE_COLS,
            out_dir,
            analysis_id=label,
            segment_length_mm=seg_mm,
            n_segments=len(seg_clean),
            k_patterns=int(n_clusters),
        )

        if not SKIP_PATTERN_FIGURES:
            fig, ax = plt.subplots(1, 2, figsize=(10, 4))
            ax[0].plot(list(K_RANGE), silhouettes, "o-")
            ax[0].axvline(n_clusters, color="green", linestyle="--", alpha=0.8, label=f"used k={n_clusters}")
            ax[0].set_xlabel("Number of clusters (k)")
            ax[0].set_ylabel("Silhouette score")
            ax[0].legend()
            ax[1].plot(list(K_RANGE), inertias, "o-")
            ax[1].axvline(n_clusters, color="green", linestyle="--", alpha=0.8, label=f"used k={n_clusters}")
            ax[1].set_xlabel("Number of clusters (k)")
            ax[1].set_ylabel("Inertia")
            ax[1].legend()
            plt.tight_layout()
            save_fig("pattern_k_selection.png")
            plt.close()
            fig, ax = plt.subplots(figsize=(10, 5))
            pt = seg_clean.groupby(["pattern", "sex"]).size().unstack(fill_value=0).reindex(sorted(seg_clean["pattern"].unique()))
            pt.index = [f"{i} ({pattern_labels.get(i, '?')})" for i in pt.index]
            pt.plot(kind="bar", stacked=True, ax=ax, color=["steelblue", "coral"], edgecolor="gray")
            ax.set_ylabel(f"Number of {seg_mm}mm segments")
            ax.set_title(f"{label}: pattern by sex (male vs female)")
            ax.legend(title="Sex")
            ax.set_xticklabels(ax.get_xticklabels(), rotation=15, ha="right")
            plt.tight_layout()
            save_fig("pattern_by_sex_stacked.png")
            plt.close()
            fig, axes = plt.subplots(1, 2, figsize=(12, 5.5))
            group_counts = seg_clean.groupby(["pattern_group", "sex"]).size().unstack(fill_value=0)
            group_counts.plot(kind="bar", stacked=True, ax=axes[0], color=["steelblue", "coral"])
            axes[0].set_title(f"{label} {seg_mm}mm: by pattern group")
            axes[0].tick_params(axis="x", rotation=15)
            seg_clean["PC1"] = pc_xy[:, 0]
            seg_clean["PC2"] = pc_xy[:, 1]
            sns.scatterplot(data=seg_clean, x="PC1", y="PC2", hue="pattern_label", alpha=0.6, ax=axes[1])
            axes[1].set_title("PCA by pattern")
            _h_pca, _lab_pca = axes[1].get_legend_handles_labels()
            axes[1].get_legend().remove()
            fig.legend(
                _h_pca,
                _lab_pca,
                bbox_to_anchor=(0.5, 0.02),
                loc="upper center",
                ncol=3,
                fontsize=8,
                title="Pattern",
                title_fontsize=9,
                frameon=True,
                columnspacing=0.9,
                handletextpad=0.4,
            )
            fig.tight_layout()
            fig.subplots_adjust(bottom=0.24)
            save_fig("pattern_pca_by_cluster_and_sex.png")
            plt.close()
            groups_sorted = seg_clean["pattern_group"].dropna().unique()
            groups_sorted = sorted(groups_sorted, key=lambda g: -(seg_clean["pattern_group"] == g).sum())
            n_groups = len(groups_sorted)
            n_cols = min(3, max(1, n_groups))
            n_rows = int(np.ceil(n_groups / n_cols)) if n_groups else 1
            fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 5 * n_rows))
            axes = np.atleast_2d(axes)
            for idx, grp in enumerate(groups_sorted):
                ax = axes.flat[idx]
                subset = seg_clean[seg_clean["pattern_group"] == grp]
                for sx in ["male", "female"]:
                    sub = subset[subset["sex"] == sx]
                    if len(sub) == 0: continue
                    for _, r in sub.iterrows():
                        s = df_seg[(df_seg["experiment_id"] == r["experiment_id"]) & (df_seg["detection_run_id"] == r["detection_run_id"]) & (df_seg["seg_40"] == r["seg_40"])].sort_values(["frame", "time_s"])
                        x = s["x"].values.astype(float) - s["x"].iloc[0]
                        y = s["y"].values.astype(float) - s["y"].iloc[0]
                        ax.plot(x, y, color="steelblue" if sx == "male" else "coral", alpha=0.25, linewidth=0.8)
                n_male = (subset["sex"] == "male").sum()
                n_female = (subset["sex"] == "female").sum()
                tit = f"{grp} (male: {n_male}, female: {n_female})"
                if "Curved" in grp:
                    tit += f"\navg turn {subset['mean_abs_turn_deg'].mean():.0f}°"
                elif "Zig-zag" in grp:
                    tit += f"\n{subset['n_turns'].mean():.1f} turns/seg"
                ax.set_title(tit)
                ax.set_xlabel("x (px from start)")
                ax.set_ylabel("y (px from start)")
                ax.axis("equal")
            for j in range(n_groups, axes.size):
                axes.flat[j].set_visible(False)
            plt.suptitle(f"{label} {seg_mm}mm (blue=male, coral=female)", y=1.02)
            plt.tight_layout()
            save_fig("all_movements_by_pattern_male_female.png")
            plt.close()

            _sex_palette = {"male": "steelblue", "female": "coral"}
            fig_b1, ax_b1 = plt.subplots(figsize=(10, 5))
            sns.boxplot(data=seg_clean, x="pattern_group", y="mean_abs_turn_deg", hue="sex", palette=_sex_palette, ax=ax_b1)
            ax_b1.set_title("Turn angle by pattern (avg °/seg)")
            ax_b1.set_xticklabels(ax_b1.get_xticklabels(), rotation=15, ha="right")
            ax_b1.legend(bbox_to_anchor=(1.02, 1), loc="upper left", borderaxespad=0, title="Sex")
            fig_b1.tight_layout(rect=[0, 0, 0.78, 1])
            save_fig("movements_avg_turn_per_pattern.png", fig_b1)
            plt.close(fig_b1)

            fig_b2, ax_b2 = plt.subplots(figsize=(10, 5))
            sns.boxplot(data=seg_clean, x="pattern_group", y="n_turns", hue="sex", palette=_sex_palette, ax=ax_b2)
            ax_b2.set_title("Number of turns by pattern")
            ax_b2.set_xticklabels(ax_b2.get_xticklabels(), rotation=15, ha="right")
            ax_b2.legend(bbox_to_anchor=(1.02, 1), loc="upper left", borderaxespad=0, title="Sex")
            fig_b2.tight_layout(rect=[0, 0, 0.78, 1])
            save_fig("movements_n_turns_per_pattern.png", fig_b2)
            plt.close(fig_b2)

            fig_s, ax_s = plt.subplots(figsize=(10, 6.5))
            sns.scatterplot(data=seg_clean, x="mean_abs_turn_deg", y="n_turns", hue="pattern_group", alpha=0.4, ax=ax_s)
            ax_s.set_title("Turn angle vs n_turns (by pattern)")
            ax_s.set_xlabel("Mean abs turn (°)")
            ax_s.legend(
                bbox_to_anchor=(0.5, -0.14),
                loc="upper center",
                ncol=3,
                fontsize=8,
                title="Pattern",
                title_fontsize=9,
                frameon=True,
                columnspacing=0.9,
                handletextpad=0.4,
            )
            fig_s.tight_layout()
            fig_s.subplots_adjust(bottom=0.24)
            save_fig("movements_turn_angle_vs_n_turns.png", fig_s)
            plt.close(fig_s)

            # Actual vial coordinates per pattern (raw x, y px; inverted y like Step 0/1/2)
            n_cols5 = min(3, n_groups)
            n_rows5 = int(np.ceil(n_groups / n_cols5))
            fig5, axes5 = plt.subplots(n_rows5, n_cols5, figsize=(6 * n_cols5, 5 * n_rows5))
            axes5 = np.atleast_2d(axes5)
            for idx_g, grp in enumerate(groups_sorted):
                ax5 = axes5.flat[idx_g]
                subset = seg_clean[seg_clean["pattern_group"] == grp]
                for sx in ["male", "female"]:
                    sub = subset[subset["sex"] == sx]
                    if len(sub) == 0:
                        continue
                    for _, r in sub.iterrows():
                        pts = df_seg[
                            (df_seg["experiment_id"] == r["experiment_id"])
                            & (df_seg["detection_run_id"] == r["detection_run_id"])
                            & (df_seg["seg_40"] == r["seg_40"])
                        ].sort_values(["frame", "time_s"])
                        ax5.plot(
                            pts["x"].values,
                            pts["y"].values,
                            color="steelblue" if sx == "male" else "coral",
                            alpha=0.30,
                            linewidth=0.8,
                        )
                n_male = (subset["sex"] == "male").sum()
                n_female = (subset["sex"] == "female").sum()
                tit = f"{grp}\nmale: {n_male}  female: {n_female}"
                if "Curved" in grp:
                    tit += f"\navg turn {subset['mean_abs_turn_deg'].mean():.0f}°"
                elif "Zig-zag" in grp:
                    tit += f"\n{subset['n_turns'].mean():.1f} turns/seg"
                ax5.set_title(tit, fontsize=9)
                ax5.set_xlabel("x (px)", fontsize=8)
                ax5.set_ylabel("y (px)", fontsize=8)
                ax5.invert_yaxis()
                ax5.set_aspect("equal", adjustable="datalim")
            for j in range(n_groups, axes5.size):
                axes5.flat[j].set_visible(False)
            plt.suptitle(
                f"{label} — actual vial positions per pattern ({seg_mm} mm)  blue=male  coral=female",
                y=1.02,
                fontsize=11,
            )
            plt.tight_layout()
            save_fig("actual_positions_by_pattern_male_female.png")
            plt.close()

            SIMILARITY_THRESHOLD = 0.4
            panels = []
            for grp in groups_sorted:
                male_sub = seg_clean[(seg_clean["pattern_group"] == grp) & (seg_clean["sex"] == "male")]
                female_sub = seg_clean[(seg_clean["pattern_group"] == grp) & (seg_clean["sex"] == "female")]
                n_male, n_female = len(male_sub), len(female_sub)
                if n_male == 0 and n_female == 0: continue
                if n_male == 0:
                    panels.append((grp, "female"))
                elif n_female == 0:
                    panels.append((grp, "male"))
                else:
                    mm_t, mf_t = male_sub["mean_abs_turn_deg"].mean(), female_sub["mean_abs_turn_deg"].mean()
                    mm_n, mf_n = male_sub["n_turns"].mean(), female_sub["n_turns"].mean()
                    rel_t = abs(mm_t - mf_t) / (max(mm_t, mf_t) + 1e-6)
                    rel_n = abs(mm_n - mf_n) / (max(mm_n, mf_n) + 1e-6)
                    if rel_t < SIMILARITY_THRESHOLD and rel_n < SIMILARITY_THRESHOLD:
                        panels.append((grp, "both"))
                    else:
                        panels.append((grp, "male"))
                        panels.append((grp, "female"))
            n_panels = len(panels)
            if n_panels > 0:
                n_cols_c = min(3, n_panels)
                n_rows_c = int(np.ceil(n_panels / n_cols_c))
                fig_c, axes_c = plt.subplots(n_rows_c, n_cols_c, figsize=(5 * n_cols_c, 5 * n_rows_c))
                axes_c = np.atleast_2d(axes_c)
                for idx, (grp, mode) in enumerate(panels):
                    ax = axes_c.flat[idx]
                    subset = seg_clean[seg_clean["pattern_group"] == grp]
                    if mode == "both":
                        for sx in ["male", "female"]:
                            sub = subset[subset["sex"] == sx]
                            if len(sub) == 0: continue
                            for _, r in sub.iterrows():
                                s = df_seg[(df_seg["experiment_id"] == r["experiment_id"]) & (df_seg["detection_run_id"] == r["detection_run_id"]) & (df_seg["seg_40"] == r["seg_40"])].sort_values(["frame", "time_s"])
                                x = s["x"].values.astype(float) - s["x"].iloc[0]
                                y = s["y"].values.astype(float) - s["y"].iloc[0]
                                ax.plot(x, y, color="steelblue" if sx == "male" else "coral", alpha=0.25, linewidth=0.8)
                        n_male = (subset["sex"] == "male").sum()
                        n_female = (subset["sex"] == "female").sum()
                        ax.set_title(f"{grp} (similar)\nmale: {n_male}, female: {n_female}")
                    else:
                        sub = subset[subset["sex"] == mode]
                        for _, r in sub.iterrows():
                            s = df_seg[(df_seg["experiment_id"] == r["experiment_id"]) & (df_seg["detection_run_id"] == r["detection_run_id"]) & (df_seg["seg_40"] == r["seg_40"])].sort_values(["frame", "time_s"])
                            x = s["x"].values.astype(float) - s["x"].iloc[0]
                            y = s["y"].values.astype(float) - s["y"].iloc[0]
                            ax.plot(x, y, color="steelblue" if mode == "male" else "coral", alpha=0.25, linewidth=0.8)
                        ax.set_title(f"{grp} ({mode} only, n={len(sub)}")
                    ax.set_xlabel("x (px from start)")
                    ax.set_ylabel("y (px from start)")
                    ax.axis("equal")
                for j in range(n_panels, axes_c.size):
                    axes_c.flat[j].set_visible(False)
                plt.suptitle(f"{label} {seg_mm}mm: comparable (same) vs individual (male/female)", y=1.02)
                plt.tight_layout()
                save_fig("pattern_comparison_male_vs_female.png")
                plt.close()

        globals()["PLOT_DIR"] = old_plot_dir
        print(f"  Saved {label} {seg_mm}mm (k={n_clusters}) -> {out_dir}")
print("Done. Step 3 (young / old, male vs female).")



  Saved 3_young 5mm (k=6) -> pattern_distance_output/3_young/5mm
  Saved 3_young 10mm (k=3) -> pattern_distance_output/3_young/10mm
  Saved 3_young 15mm (k=4) -> pattern_distance_output/3_young/15mm
  Saved 3_young 20mm (k=4) -> pattern_distance_output/3_young/20mm
  Saved 3_young 30mm (k=5) -> pattern_distance_output/3_young/30mm
  Saved 3_young 40mm (k=5) -> pattern_distance_output/3_young/40mm
  Saved 3_old 5mm (k=2) -> pattern_distance_output/3_old/5mm
  Saved 3_old 10mm (k=3) -> pattern_distance_output/3_old/10mm
  Saved 3_old 15mm (k=3) -> pattern_distance_output/3_old/15mm
  Saved 3_old 20mm (k=4) -> pattern_distance_output/3_old/20mm
  Saved 3_old 30mm (k=3) -> pattern_distance_output/3_old/30mm
  Saved 3_old 40mm (k=3) -> pattern_distance_output/3_old/40mm
Done. Step 3 (young / old, male vs female).


## Pooled `sd_09_08` + `sd_09_09`: w1118 vs dehydrated w1118 (Sign-up list.pdf)

**SD 9-9-25** (`sd_09_09/`): videos **1, 3, 5 = w1118**; **2, 4, 6 = dehydrated w1118** → `detection_output_<N>.csv` or `detection_output_<N>_filtered.csv`.

**SD 9-8-25** (`sd_09_08/`): videos **1–3 = On media**; **4–7 = Dehydrated** → same file naming.

The code cell runs the **same Step 0–style pipeline** as the rest of the notebook: for each distance (5, 10, 15, 20, 30, 40 mm) it writes **eight PNGs** under `pattern_distance_output/w1118_dehydration_step0/<N>mm/` (the usual Step 0 figures plus `step0_movements_avg_turn_per_pattern.png`, `step0_movements_n_turns_per_pattern.png`, `step0_movements_turn_angle_vs_n_turns.png`) plus `step0_pattern_summary.csv`. Blue = w1118 on media, coral = dehydrated (same hue convention as young/old in Step 0).

Implementation: `pattern_dehydration_step0.py` (imported by the notebook).


In [9]:
# Superseded — use the **next cell**: it reads raw `detection_output_*.csv` from `sd_09_08/` and `sd_09_09/`, applies the PDF mapping, and saves plots under `pattern_distance_output/w1118_dehydration/`.
pass


In [10]:
# --- w1118 vs dehydrated: read individual detection_output_*.csv (sd_09_08, sd_09_09) ---
# No merged_detection_all.csv. Expect folders next to this notebook (same layout as merge_detection_csvs.py):
#   .../sd_09_08/detection_output_<N>.csv or detection_output_<N>_filtered.csv
#   .../sd_09_09/detection_output_<N>.csv or detection_output_<N>_filtered.csv
# Requires the **imports** cell: np, pd, sns, plt, Path, Optional, DEFAULT_MM_PER_PX_*,
# MM_PER_PX_BY_RUN, PLOT_DIR, save_fig.

import re

from IPython.display import display, Markdown

FILENAME_RE = re.compile(
    r"detection_output_(\d+)(?:_filtered)?\.csv$", re.IGNORECASE
)

# Folders sd_09_08 / sd_09_09 live here (change if your raw CSVs are elsewhere).
DEHYDRATION_DATA_BASE = Path.cwd()

PDF_NOTES = r"""
### Sign-up list.pdf — which `detection_output_N.csv` is which

**SD 9-9-25** (folder `sd_09_09`, experiment id `sd_09_09`) — *Dehydration WIG Males and controls*  
PDF notes: **Videos 1, 3, 5: w1118** · **Videos 2, 4, 6: dehydrated w1118**

| CSV file | Condition (PDF wording) |
|----------|-------------------------|
| `detection_output_1.csv` | **w1118** (on media) |
| `detection_output_2.csv` | **dehydrated w1118** |
| `detection_output_3.csv` | **w1118** (on media) |
| `detection_output_4.csv` | **dehydrated w1118** |
| `detection_output_5.csv` | **w1118** (on media) |
| `detection_output_6.csv` | **dehydrated w1118** |

---

**SD 9-8-25** (folder `sd_09_08`, experiment id `sd_09_08`) — *Dehydration WIG Males and controls*  
PDF notes: **Videos 1–3: On media** · **Videos 4–7: Dehydrated**  
(We plot these as **w1118 / on-media** vs **w1118 dehydrated** to match the 9-9 cohort design.)

| CSV file | Condition (PDF wording) |
|----------|-------------------------|
| `detection_output_1.csv` … `detection_output_3.csv` | **On media** → pooled as **w1118 (on media)** |
| `detection_output_4.csv` … `detection_output_7.csv` | **Dehydrated** → pooled as **w1118 dehydrated** |

**N** is the video index (same as **`detection_run_id`**). Files may be named `detection_output_N.csv` or `detection_output_N_filtered.csv`.
"""
display(Markdown(PDF_NOTES))

POOL_EXPS = ("sd_09_08", "sd_09_09")


def signup_w1118_condition(exp: str, detection_run_id: int) -> Optional[str]:
    e, r = str(exp), int(detection_run_id)
    if e == "sd_09_09":
        if r in (1, 3, 5):
            return "w1118 (on media)"
        if r in (2, 4, 6):
            return "w1118 dehydrated"
        return None
    if e == "sd_09_08":
        if r in (1, 2, 3):
            return "w1118 (on media)"
        if r in (4, 5, 6, 7):
            return "w1118 dehydrated"
        return None
    return None


rows = []
for exp in POOL_EXPS:
    n_vid = 7 if exp == "sd_09_08" else 6
    for vid in range(1, n_vid + 1):
        rows.append((exp, vid, signup_w1118_condition(exp, vid)))
signup_tbl = pd.DataFrame(
    rows, columns=["experiment_id", "detection_run_id (video #)", "condition (from PDF)"]
)
display(signup_tbl)

pdf_csv_map = pd.DataFrame(
    [
        ("sd_09_09", 1, "detection_output_1.csv", "w1118 (PDF: video 1)"),
        ("sd_09_09", 2, "detection_output_2.csv", "dehydrated w1118 (PDF: video 2)"),
        ("sd_09_09", 3, "detection_output_3.csv", "w1118 (PDF: video 3)"),
        ("sd_09_09", 4, "detection_output_4.csv", "dehydrated w1118 (PDF: video 4)"),
        ("sd_09_09", 5, "detection_output_5.csv", "w1118 (PDF: video 5)"),
        ("sd_09_09", 6, "detection_output_6.csv", "dehydrated w1118 (PDF: video 6)"),
        ("sd_09_08", 1, "detection_output_1.csv", "on media (PDF) → w1118 (on media)"),
        ("sd_09_08", 2, "detection_output_2.csv", "on media (PDF) → w1118 (on media)"),
        ("sd_09_08", 3, "detection_output_3.csv", "on media (PDF) → w1118 (on media)"),
        ("sd_09_08", 4, "detection_output_4.csv", "Dehydrated (PDF) → w1118 dehydrated"),
        ("sd_09_08", 5, "detection_output_5.csv", "Dehydrated (PDF) → w1118 dehydrated"),
        ("sd_09_08", 6, "detection_output_6.csv", "Dehydrated (PDF) → w1118 dehydrated"),
        ("sd_09_08", 7, "detection_output_7.csv", "Dehydrated (PDF) → w1118 dehydrated"),
    ],
    columns=["experiment_id", "N (video #)", "CSV filename", "PDF → plot group"],
)
print("PDF → which CSV (same rules as `signup_w1118_condition`):")
display(pdf_csv_map)


def load_raw_detection_pool(base: Path, experiment_ids: tuple[str, ...]) -> pd.DataFrame:
    """Concatenate raw detection CSVs; add experiment_id + detection_run_id from path."""
    parts: list[pd.DataFrame] = []
    for exp in experiment_ids:
        exp_dir = base / exp
        if not exp_dir.is_dir():
            print(f"Skip (folder missing): {exp_dir}")
            continue
        for csv_path in sorted(exp_dir.glob("detection_output_*.csv")):
            m = FILENAME_RE.search(csv_path.name)
            if not m:
                continue
            run_id = int(m.group(1))
            chunk = pd.read_csv(csv_path, low_memory=False)
            req = {"frame", "time_s", "x", "y"}
            miss = req - set(chunk.columns)
            if miss:
                raise ValueError(f"{csv_path}: missing columns {sorted(miss)}")
            chunk = chunk.copy()
            chunk["experiment_id"] = exp
            chunk["detection_run_id"] = run_id
            parts.append(chunk)
    if not parts:
        return pd.DataFrame()
    return pd.concat(parts, ignore_index=True)


print(f"Reading individual CSVs under {DEHYDRATION_DATA_BASE.resolve()}")
sub = load_raw_detection_pool(DEHYDRATION_DATA_BASE, POOL_EXPS)

if sub.empty:
    print(
        "No matching detection CSVs. Expected `detection_output_<N>.csv` or "
        "`detection_output_<N>_filtered.csv` under:\n"
        f"  {DEHYDRATION_DATA_BASE / 'sd_09_08'}/\n"
        f"  {DEHYDRATION_DATA_BASE / 'sd_09_09'}/\n"
        "Or set DEHYDRATION_DATA_BASE to the parent folder that contains those two subfolders."
    )
else:
    video_cols = ["experiment_id", "detection_run_id"]
    vkeys = sub[video_cols].drop_duplicates().reset_index(drop=True)
    mmx, mmy = [], []
    for _, r in vkeys.iterrows():
        key = (str(r["experiment_id"]), int(r["detection_run_id"]))
        tx, ty = MM_PER_PX_BY_RUN.get(key, (DEFAULT_MM_PER_PX_X, DEFAULT_MM_PER_PX_Y))
        mmx.append(tx)
        mmy.append(ty)
    vkeys["mm_per_px_x"] = mmx
    vkeys["mm_per_px_y"] = mmy
    sub = sub.merge(vkeys, on=video_cols, how="left")
    sub = sub.sort_values(video_cols + ["frame", "time_s"]).reset_index(drop=True)

    g = sub.groupby(video_cols, sort=False)
    sub["dt_s"] = g["time_s"].diff()
    sub["dx_px"] = g["x"].diff()
    sub["dy_px"] = g["y"].diff()
    sub.loc[sub["dt_s"] == 0, "dt_s"] = np.nan
    dx_mm = sub["dx_px"] * sub["mm_per_px_x"]
    dy_mm = sub["dy_px"] * sub["mm_per_px_y"]
    sub["step_dist_mm"] = np.sqrt(dx_mm**2 + dy_mm**2).fillna(0.0)
    sub["speed_mm_s"] = sub["step_dist_mm"] / sub["dt_s"]
    sub["cum_dist_mm"] = g["step_dist_mm"].cumsum()
    total = (
        sub.groupby(video_cols, as_index=False)["cum_dist_mm"]
        .max()
        .rename(columns={"cum_dist_mm": "total_dist_mm"})
    )
    sub = sub.merge(total, on=video_cols, how="left")
    print(f"Rows: {len(sub):,}  videos: {sub[video_cols].drop_duplicates().shape[0]}")

    sub["condition"] = [
        signup_w1118_condition(e, r) for e, r in zip(sub["experiment_id"], sub["detection_run_id"])
    ]
    ok = sub[sub["condition"].notna()].copy()
    miss = sub[sub["condition"].isna()][["experiment_id", "detection_run_id"]].drop_duplicates()
    if len(miss):
        print("Videos outside PDF lists (still plotted if any rows):")
        display(miss)

    agg = dict(
        mean_speed_mm_s=("speed_mm_s", lambda s: np.nanmean(s.values)),
        total_dist_mm=("total_dist_mm", "first"),
        n_rows=("frame", "count"),
    )
    if "w" in ok.columns:
        agg["mean_w_px"] = ("w", lambda s: np.nanmean(s.values))

    per_video = ok.groupby(["experiment_id", "detection_run_id", "condition"], as_index=False).agg(**agg)
    per_video["pooled"] = np.where(
        per_video["condition"].str.contains("dehydr", case=False, na=False),
        "dehydrated w1118",
        "w1118 on media",
    )
    display(per_video.sort_values(["experiment_id", "detection_run_id"]))

    # Same five figures per distance as Step 0 global (see pattern_dehydration_step0.py)
    import importlib
    import pattern_dehydration_step0 as _pd_step
    importlib.reload(_pd_step)
    from pattern_dehydration_step0 import run_dehydration_step0_style

    df_w = ok.copy()
    df_w["category"] = df_w["condition"].astype(str)
    df_w["age_group"] = np.where(
        df_w["condition"].astype(str).str.contains("dehydr", case=False, na=False),
        "old",
        "young",
    )
    BASE_W = Path("pattern_distance_output") / "w1118_dehydration_step0"
    sum_w = run_dehydration_step0_style(
        df_w,
        video_cols,
        FEATURE_COLS,
        turn_stats_from_xy,
        BASE_W,
        hue_display_names=("w1118", "Dehydrated w1118"),
        title_prefix="W1118 dehydration (sd_09_08 + sd_09_09)",
        skip_figures=SKIP_PATTERN_FIGURES,
        only_cluster_naming_order_csv=ONLY_CLUSTER_NAMING_ORDER_CSV,
    )
    display(sum_w)
    print("Step 0–style outputs (8 PNGs each):", BASE_W.resolve())



### Sign-up list.pdf — which `detection_output_N.csv` is which

**SD 9-9-25** (folder `sd_09_09`, experiment id `sd_09_09`) — *Dehydration WIG Males and controls*  
PDF notes: **Videos 1, 3, 5: w1118** · **Videos 2, 4, 6: dehydrated w1118**

| CSV file | Condition (PDF wording) |
|----------|-------------------------|
| `detection_output_1.csv` | **w1118** (on media) |
| `detection_output_2.csv` | **dehydrated w1118** |
| `detection_output_3.csv` | **w1118** (on media) |
| `detection_output_4.csv` | **dehydrated w1118** |
| `detection_output_5.csv` | **w1118** (on media) |
| `detection_output_6.csv` | **dehydrated w1118** |

---

**SD 9-8-25** (folder `sd_09_08`, experiment id `sd_09_08`) — *Dehydration WIG Males and controls*  
PDF notes: **Videos 1–3: On media** · **Videos 4–7: Dehydrated**  
(We plot these as **w1118 / on-media** vs **w1118 dehydrated** to match the 9-9 cohort design.)

| CSV file | Condition (PDF wording) |
|----------|-------------------------|
| `detection_output_1.csv` … `detection_output_3.csv` | **On media** → pooled as **w1118 (on media)** |
| `detection_output_4.csv` … `detection_output_7.csv` | **Dehydrated** → pooled as **w1118 dehydrated** |

**N** is the video index (same as **`detection_run_id`**). Files may be named `detection_output_N.csv` or `detection_output_N_filtered.csv`.


,experiment_id,detection_run_id (video #),condition (from PDF)
0,sd_09_08,1,w1118 (on media)
1,sd_09_08,2,w1118 (on media)
2,sd_09_08,3,w1118 (on media)
3,sd_09_08,4,w1118 dehydrated
4,sd_09_08,5,w1118 dehydrated
5,sd_09_08,6,w1118 dehydrated
6,sd_09_08,7,w1118 dehydrated
7,sd_09_09,1,w1118 (on media)
8,sd_09_09,2,w1118 dehydrated
9,sd_09_09,3,w1118 (on media)


PDF → which CSV (same rules as `signup_w1118_condition`):


,experiment_id,N (video #),CSV filename,PDF → plot group
0,sd_09_09,1,detection_output_1.csv,w1118 (PDF: video 1)
1,sd_09_09,2,detection_output_2.csv,dehydrated w1118 (PDF: video 2)
2,sd_09_09,3,detection_output_3.csv,w1118 (PDF: video 3)
3,sd_09_09,4,detection_output_4.csv,dehydrated w1118 (PDF: video 4)
4,sd_09_09,5,detection_output_5.csv,w1118 (PDF: video 5)
5,sd_09_09,6,detection_output_6.csv,dehydrated w1118 (PDF: video 6)
6,sd_09_08,1,detection_output_1.csv,on media (PDF) → w1118 (on media)
7,sd_09_08,2,detection_output_2.csv,on media (PDF) → w1118 (on media)
8,sd_09_08,3,detection_output_3.csv,on media (PDF) → w1118 (on media)
9,sd_09_08,4,detection_output_4.csv,Dehydrated (PDF) → w1118 dehydrated


Reading individual CSVs under /Users/yashpatel/Documents/John-Tower-Lab/paper_trajectory/Exp1
Skip (folder missing): /Users/yashpatel/Documents/John-Tower-Lab/paper_trajectory/Exp1/sd_09_08
Skip (folder missing): /Users/yashpatel/Documents/John-Tower-Lab/paper_trajectory/Exp1/sd_09_09
No matching detection CSVs. Expected `detection_output_<N>.csv` or `detection_output_<N>_filtered.csv` under:
  /Users/yashpatel/Documents/John-Tower-Lab/paper_trajectory/Exp1/sd_09_08/
  /Users/yashpatel/Documents/John-Tower-Lab/paper_trajectory/Exp1/sd_09_09/
Or set DEHYDRATION_DATA_BASE to the parent folder that contains those two subfolders.
